# Optimisation de profit d'un atelier en utilisant QAOA

## Table des matieres

- [Exposition du probleme](#Exposition-du-probleme)
- [Translation mathematique du probleme](#Translation-mathematique-du-probleme)
- [Definition des variables](#Definition-des-variables)
- [Resolution du probleme (Recherche exhaustive)](#Resolution-du-probleme-%28Recherche-exhaustive%29)
    - [Resultat de la recherche exhaustive](#Resultat-de-la-recherche-exhaustive)
    - [Resultat de la recherche exhaustive (x2 commandes)](#Resultat-de-la-recherche-exhaustive-%28x2-commandes%29)
- [Resolution du probleme (QAOA)](#Resolution-du-probleme-%28QAOA%29)
    - [Modele mathematique](#Modele-mathematique)
    - [Optimisation Binaire Quadratique sans Contrainte (Quadratic unconstrained binary optimization / QUBO)](#Optimisation-Binaire-Quadratique-sans-Contrainte-%28Quadratic-unconstrained-binary-optimization-%2F-QUBO%29)
    - [Hamiltonien](#Hamiltonien)
    - [Boucle hybride](#Boucle-hybride)
    - [Implementation](#Implementation)
        - [Etape 1 : le QUBO](#Etape-1-:-le-QUBO)
        - [Etape 2 : le Hamiltonien](#Etape-2-:-le-Hamiltonien)
        - [Etape 3 : le circuit](#Etape-3-:-le-circuit)
        - [Etape 4 : la boucle hybride](#Etape-4-:-la-boucle-hybride)
        - [Etape 5 : echantillonner et garder la meilleure selection valide](#Etape-5-:-echantillonner-et-garder-la-meilleure-selection-valide)
    - [Benchmark de QAOA](#Benchmark-de-QAOA)
- [Interpretation des resultats](#Interpretation-des-resultats)
    - [Algorithme classique contre Algorithme mixte](#Algorithme-classique-contre-Algorithme-mixte)
    - [Affirmations basees sur notre exemple d'atelier](#Affirmations-basees-sur-notre-exemple-d'atelier)
- [Conclusion (QAOA sur emulateur)](#Conclusion-%28QAOA-sur-emulateur%29)
- [Execution sur une machine IBM](#Execution-sur-une-machine-IBM)
- [Conclusion](#Conclusion)
- [Glossaire](#Glossaire)



### Fonctionnement du notebook:
Ce notebook contient des textes d'explications ainsi que des PoC ecrits en python.
Les blocks python se distinguent en 2 categories:
1. Les definitions de fonctions, qui sont reutilisees plus tard
2. Le travail qui utilise ces fonctions.
 
Pour les lecteurs souhaitant sauter certaines sections, il est necessaire d'executer tout les blocks contenant la mention `#DEFINITION_RUN`. Un Ctrl-F vous permet de les lister.
N'importe quelle block peut s'executer si tout les definitions `#DEFINITION_RUN` ont ete execute.

## Exposition du probleme

Un atelier possede certaines **resources**:

- Machines
- Ouvriers
- Creneaux horraires

Un atelier doit prendre une **decision**:

- A quelles commandes repondre ? 

Un atelier a un objectif:

- Un atelier doit choisir ses commandes pour **maximiser ses profits**

Un atelier a une regle:

- Il est impossible de repondre a 2 commandes necesitant les meme resources.

## Translation mathematique du probleme

Ce probleme correspond au [Problème du stable maximum](https://fr.wikipedia.org/wiki/Probl%C3%A8me_du_stable_maximum) avec poids (**Maximum Weight Independent Set / MWIS**):

> En informatique théorique, le problème du stable maximum ou maximum independent set problem en anglais, est un problème d'optimisation qui consiste étant donné un graphe non orienté à trouver un stable de cardinal maximum, c'est-à-dire un sous-ensemble de sommets du graphe, le plus grand possible, tel que les éléments de ce sous-ensemble ne soient pas voisins.
>> https://fr.wikipedia.org/wiki/Probl%C3%A8me_du_stable_maximum

Tout les probleme MWIS sont **NP-Hard**:    
il y a $2^n$ candidats pour n commandes, puisque chaque commande peut avoir un des 2 etats booleens: acceptee ou refusee.    
Une commande sera acceptee si elle maximise le profit et ne cree pas dans de conflict.

## Definition des variables

Pour notre exemple, nous allons definir un atelier tel que:

- 4 machines
- 3 ouvriers
- 3 creneaux horraires

Qui doivent repondre a 8 commandes.

Voici les definitions du probleme en python:

In [1]:
#DEFINITION_RUN
# Tableau 1 :
# les ressources de l'atelier (nom -> type).
# Chacune a une capacité de 1 commande a la fois.
RESSOURCES: dict[str, str] = {
    "Fraiseuse CNC":      "machine",
    "Découpeuse laser":   "machine",
    "Cabine de peinture": "machine",
    "Tour":               "machine",
    "Soudeur certifié":   "ouvrier",
    "Inspecteur qualité": "ouvrier",
    "Électricien":        "ouvrier",
    "Créneau lundi":      "créneau",
    "Créneau mercredi":   "créneau",
    "Créneau vendredi":   "créneau",
}
print("Ressources:", RESSOURCES)

Ressources: {'Fraiseuse CNC': 'machine', 'Découpeuse laser': 'machine', 'Cabine de peinture': 'machine', 'Tour': 'machine', 'Soudeur certifié': 'ouvrier', 'Inspecteur qualité': 'ouvrier', 'Électricien': 'ouvrier', 'Créneau lundi': 'créneau', 'Créneau mercredi': 'créneau', 'Créneau vendredi': 'créneau'}


In [2]:
#DEFINITION_RUN
# Tableau 2 :
# La liste de profit des commandes.
# La commande n a pour profit COMMANDES[n - 1]
# Il y a autant de commande que d'element dans ce tableau.
COMMANDES: list[int] = [
    5, # Commande 1 a pour profit 5
    3,
    4,
    6,
    2,
    5,
    3,
    4
]
print(f"{len(COMMANDES)} commandes au profit:", COMMANDES)

8 commandes au profit: [5, 3, 4, 6, 2, 5, 3, 4]


In [3]:
#DEFINITION_RUN
# Tableau 3 : les conflits (ressource -> commandes qui en ont besoin).
# Deux commandes de la même ligne ne peuvent pas être acceptées ensemble.
CONFLITS: dict[str, list[int]] = {
    "Fraiseuse CNC":      [1, 2],
    "Découpeuse laser":   [3, 4],
    "Cabine de peinture": [5, 6],
    "Tour":               [7, 8],
    "Soudeur certifié":   [1, 3],
    "Inspecteur qualité": [4, 6],
    "Électricien":        [5, 7],
    "Créneau lundi":      [2, 4],
    "Créneau mercredi":   [3, 5],
    "Créneau vendredi":   [6, 8],
}

def conflits_pretty_print() -> None:
    largeur_nom: int = max(map(len, CONFLITS))
    largeur_type: int = max(map(len, RESSOURCES.values()))
    for ressource, commandes in CONFLITS.items():
        liste: str = " et ".join(f"n°{c} (profit {COMMANDES[c - 1]})" for c in commandes)
        print(f"{ressource:<{largeur_nom}}  [{RESSOURCES[ressource]:<{largeur_type}}]  ->  {liste}")

In [4]:
conflits_pretty_print()

Fraiseuse CNC       [machine]  ->  n°1 (profit 5) et n°2 (profit 3)
Découpeuse laser    [machine]  ->  n°3 (profit 4) et n°4 (profit 6)
Cabine de peinture  [machine]  ->  n°5 (profit 2) et n°6 (profit 5)
Tour                [machine]  ->  n°7 (profit 3) et n°8 (profit 4)
Soudeur certifié    [ouvrier]  ->  n°1 (profit 5) et n°3 (profit 4)
Inspecteur qualité  [ouvrier]  ->  n°4 (profit 6) et n°6 (profit 5)
Électricien         [ouvrier]  ->  n°5 (profit 2) et n°7 (profit 3)
Créneau lundi       [créneau]  ->  n°2 (profit 3) et n°4 (profit 6)
Créneau mercredi    [créneau]  ->  n°3 (profit 4) et n°5 (profit 2)
Créneau vendredi    [créneau]  ->  n°6 (profit 5) et n°8 (profit 4)


Ici, les verifications des donnees et la creation des `PAIRES` de conflits:

In [5]:
#DEFINITION_RUN
# Assertion des listes et des creation de la liste de conflits:
from itertools import combinations

# Vérifie que les trois tableaux parlent des mêmes ressources et commandes.
def verifier_coherence(
    ressources: dict[str, str],
    commandes: list[int],
    conflits: dict[str, list[int]],
) -> None:
    assert set(conflits) == set(ressources), "ressource inconnue ou oubliée dans CONFLITS"
    n: int = len(commandes)
    for ressource, numeros in conflits.items():
        for c in numeros:
            assert 1 <= c <= n, f"commande inconnue : {c} (ressource {ressource})"

# Retourne les paires de commandes qui ne peuvent pas être acceptées ensemble.
def calculer_paires(conflits: dict[str, list[int]]) -> list[tuple[int, int]]:
    paires: list[tuple[int, int]] = []
    for numeros in conflits.values():
        for paire in combinations(sorted(numeros), 2):
            if paire not in paires:  # évite un doublon si deux commandes partagent 2 ressources
                paires.append(paire)
    return sorted(paires)

# Retourne les ressources dont la commande c a besoin
def ressources_de_commande(c: int, conflits: dict[str, list[int]]) -> list[str]:
    return [ressource for ressource, numeros in conflits.items() if c in numeros]

# Affiche pour chaque commande son profit et ses ressources.
def afficher_commandes(commandes: list[int], conflits: dict[str, list[int]]) -> None:
    for c in range(1, len(commandes) + 1):
        besoins: str = ", ".join(ressources_de_commande(c, conflits))
        print(f"Commande {c} (profit {commandes[c - 1]}) : {besoins}")

N: int = len(COMMANDES)
PAIRES: list[tuple[int, int]] = calculer_paires(CONFLITS)

In [6]:
# Verifie les tableaux et affiche les besoins de chaque commande.
verifier_coherence(RESSOURCES, COMMANDES, CONFLITS)
afficher_commandes(COMMANDES, CONFLITS)
print(f"\n{len(PAIRES)} paires en conflit : {PAIRES}")

Commande 1 (profit 5) : Fraiseuse CNC, Soudeur certifié
Commande 2 (profit 3) : Fraiseuse CNC, Créneau lundi
Commande 3 (profit 4) : Découpeuse laser, Soudeur certifié, Créneau mercredi
Commande 4 (profit 6) : Découpeuse laser, Inspecteur qualité, Créneau lundi
Commande 5 (profit 2) : Cabine de peinture, Électricien, Créneau mercredi
Commande 6 (profit 5) : Cabine de peinture, Inspecteur qualité, Créneau vendredi
Commande 7 (profit 3) : Tour, Électricien
Commande 8 (profit 4) : Tour, Créneau vendredi

10 paires en conflit : [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]


## Resolution du probleme (Recherche exhaustive)

En se basant sur les tableaux et valeurs d'exemples definis precedement, nous pouvons resoudre le probleme avec un algorithme exhaustif qui itere sur les 2^n possibilites:

In [7]:
#DEFINITION_RUN
from itertools import product
from typing import NamedTuple

class Resultat(NamedTuple):
    acceptees: list[int]  # numéros (à partir de 1) des commandes acceptées
    profit: int
    nb_testes: int        # choix examinés : 2^N
    nb_valides: int       # choix sans aucun conflit

# Un choix est une suite de 0 et de 1 : choix[i - 1] vaut 1 si la commande i est acceptée.
# Il est valide si aucune paire en conflit n'est acceptée en même temps.
def est_valide(choix: tuple[int, ...], paires: list[tuple[int, int]]) -> bool:
    return all(not (choix[i - 1] and choix[j - 1]) for i, j in paires)

# Somme des profits des commandes acceptées.
def calculer_profit(choix: tuple[int, ...], commandes: list[int]) -> int:
    return sum(profit for profit, accepte in zip(commandes, choix) if accepte)

# Teste les 2^N choix possibles et garde le valide qui rapporte le plus.
def recherche_exhaustive(commandes: list[int], paires: list[tuple[int, int]]) -> Resultat:
    meilleur_choix: tuple[int, ...] = (0,) * len(commandes)
    meilleur_profit: int = 0
    nb_testes: int = 0
    nb_valides: int = 0
    for choix in product((0, 1), repeat=len(commandes)):
        nb_testes += 1
        if not est_valide(choix, paires):
            continue
        nb_valides += 1
        profit: int = calculer_profit(choix, commandes)
        if profit > meilleur_profit:
            meilleur_choix, meilleur_profit = choix, profit
    acceptees: list[int] = [i + 1 for i, x in enumerate(meilleur_choix) if x]
    return Resultat(acceptees, meilleur_profit, nb_testes, nb_valides)

print("Defini:", est_valide, calculer_profit, recherche_exhaustive)

Defini: <function est_valide at 0x7e97a2525760> <function calculer_profit at 0x7e97a2526de0> <function recherche_exhaustive at 0x7e97a2527240>


In [8]:
#DEFINITION_RUN
# Modèle et fréquence du CPU : platform.processor() ne donne souvent que l'architecture.
def modele_cpu() -> str:
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as f:
            infos: dict[str, str] = {}
            for ligne in f:
                cle, _, valeur = ligne.partition(":")
                infos.setdefault(cle.strip(), valeur.strip())
        modele: str = infos.get("model name", platform.processor() or "inconnu")
        return f"{modele} @ {float(infos['cpu MHz']):.0f} MHz" if "cpu MHz" in infos else modele
    except OSError:  # pas de /proc hors Linux
        return platform.processor() or "inconnu"

# Affiche la machine et le processus qui lancent la mesure, pour pouvoir comparer les durées.
def print_running_env() -> None:
    try:
        ram_go: str = f"{os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3:.1f} Go"
    except (ValueError, OSError, AttributeError):  # indisponible hors Unix
        ram_go = "inconnue"
    try:
        cpus_utilisables: int = len(os.sched_getaffinity(0))
    except AttributeError:  # absent sous macOS / Windows
        cpus_utilisables = os.cpu_count() or 0
    print("== Benchmark ==")
    print(f"Date           : {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"Machine        : {platform.node()} ({platform.system()} {platform.release()}, {platform.machine()})")
    print(f"Processeur     : {modele_cpu()}")
    print(f"Coeurs         : {os.cpu_count()} logiques ({cpus_utilisables} utilisables)")
    if hasattr(os, "getloadavg"):
        charge: tuple[float, float, float] = os.getloadavg()
        print(f"Charge système : {charge[0]:.2f} / {charge[1]:.2f} / {charge[2]:.2f} (1 / 5 / 15 min)")
    print(f"Mémoire        : {ram_go}")
    print(f"Python         : {platform.python_implementation()} {platform.python_version()} ({sys.executable})")
    print("===============")

print("Defini:", modele_cpu, print_running_env)

Defini: <function modele_cpu at 0x7e97a2527560> <function print_running_env at 0x7e97a2525bc0>


In [9]:
#DEFINITION_RUN
# Mesure de la recherche exhaustive : duree, temps CPU, memoire, changements de contexte.
import os
import platform
import resource
import sys
from datetime import datetime
from time import perf_counter

# Exécute la force brute `repetitions` fois et retourne le résultat avec la durée moyenne (en secondes).
def mesurer_recherche_exhaustive(
    commandes: list[int], paires: list[tuple[int, int]], repetitions: int
) -> tuple[Resultat, float]:
    assert repetitions >= 1, "repetitions doit être au moins 1"
    durees: list[float] = []
    for _ in range(repetitions):
        debut: float = perf_counter()
        resultat: Resultat = recherche_exhaustive(commandes, paires)
        durees.append(perf_counter() - debut)
    return resultat, sum(durees) / repetitions

def print_variables_in_use(
    commandes: list[int], paires: list[tuple[int, int]], ressources: dict[str, str] | None = None
) -> None:
    ressources = RESSOURCES if ressources is None else ressources
    print("== Variables ==")
    print(f"{len(ressources)} ressources: {ressources}")
    print(f"Profits des {len(commandes)} commandes: {commandes}")
    print(f"Paires des {len(paires)} conflits: {paires}")
    print("===============")

# Retourne l'usage CPU moyen par exécution : temps réel, CPU et changements de contexte divisés par `repetitions`.
# Les informations de la machine montrent que la recherche exhaustive et QAOA sont mesures sur le meme systeme.
def mesurer_avec_usage_cpu(
    commandes: list[int], paires: list[tuple[int, int]], repetitions: int, display: bool,
    ressources: dict[str, str] | None = None,
) -> tuple[Resultat, float, dict[str, float]]:
    print_variables_in_use(commandes, paires, ressources)
    avant = resource.getrusage(resource.RUSAGE_SELF)
    debut: float = perf_counter()
    resultat, duree_moyenne = mesurer_recherche_exhaustive(commandes, paires, repetitions)
    reel: float = perf_counter() - debut
    apres = resource.getrusage(resource.RUSAGE_SELF)
    utilisateur: float = apres.ru_utime - avant.ru_utime
    systeme: float = apres.ru_stime - avant.ru_stime
    usage: dict[str, float] = {
        "reel": reel / repetitions,
        "utilisateur": utilisateur / repetitions,
        "systeme": systeme / repetitions,
        "taux_cpu": 100 * (utilisateur + systeme) / reel,  # rapport de cumuls, identique à la moyenne
        "pic_memoire_mo": apres.ru_maxrss / 1024,  # pic global : un maximum ne se moyenne pas
        "ctx_volontaires": (apres.ru_nvcsw - avant.ru_nvcsw) / repetitions,
        "ctx_forces": (apres.ru_nivcsw - avant.ru_nivcsw) / repetitions,
    }
    if display:
        print(f"Durée moyenne sur {repetitions} exécutions : {duree_moyenne * 1000:.3f} ms")
        print(f"{resultat.nb_testes} possibilités testées par exécution")
        print(f"{resultat.nb_valides} resultats valides ({100 * resultat.nb_valides / resultat.nb_testes:.5f} %)")
        print(f"Meilleure sélection : commandes {resultat.acceptees}, profit {resultat.profit}")
        print(f"Temps réel moyen : {usage['reel'] * 1000:.3f} ms")
        print(f"Temps CPU moyen  : {usage['utilisateur'] * 1000:.3f} ms utilisateur + {usage['systeme'] * 1000:.3f} ms système ({usage['taux_cpu']:.0f} % d'un coeur)")
        print(f"Pic de mémoire   : {usage['pic_memoire_mo']:.1f} Mo")
        print(f"Changements de contexte moyens : {usage['ctx_volontaires']:.4f} volontaires, {usage['ctx_forces']:.4f} forcés par exécution")
    return resultat, duree_moyenne, usage

print("Defini:", mesurer_recherche_exhaustive, print_variables_in_use, mesurer_avec_usage_cpu)

Defini: <function mesurer_recherche_exhaustive at 0x7e97a25274c0> <function print_variables_in_use at 0x7e97a25277e0> <function mesurer_avec_usage_cpu at 0x7e97a2527c40>


In [10]:
# Execution des fonctions de recherches exhaustives
# Avec monitoring pour pouvoir mesurer les resultats.
repetitions=10000
print_running_env()
mesurer_avec_usage_cpu(COMMANDES, PAIRES, repetitions=repetitions, display=True)

== Benchmark ==
Date           : 2026-11-24 22:38:23
Machine        : RulianPC (Linux 6.18.33.2-microsoft-standard-WSL2, x86_64)
Processeur     : Intel(R) Core(TM) Ultra 7 155H @ 2995 MHz
Coeurs         : 22 logiques (22 utilisables)
Charge système : 0.29 / 0.39 / 0.53 (1 / 5 / 15 min)
Mémoire        : 14.6 Go
Python         : CPython 3.12.3 (/home/julian/Quantum-Computing/.venv/bin/python)
== Variables ==
10 ressources: {'Fraiseuse CNC': 'machine', 'Découpeuse laser': 'machine', 'Cabine de peinture': 'machine', 'Tour': 'machine', 'Soudeur certifié': 'ouvrier', 'Inspecteur qualité': 'ouvrier', 'Électricien': 'ouvrier', 'Créneau lundi': 'créneau', 'Créneau mercredi': 'créneau', 'Créneau vendredi': 'créneau'}
Profits des 8 commandes: [5, 3, 4, 6, 2, 5, 3, 4]
Paires des 10 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]


Durée moyenne sur 10000 exécutions : 0.123 ms
256 possibilités testées par exécution
41 resultats valides (16.01562 %)
Meilleure sélection : commandes [1, 4, 5, 8], profit 17
Temps réel moyen : 0.124 ms
Temps CPU moyen  : 0.118 ms utilisateur + 0.000 ms système (96 % d'un coeur)
Pic de mémoire   : 67.9 Mo
Changements de contexte moyens : 0.0036 volontaires, 0.0010 forcés par exécution


(Resultat(acceptees=[1, 4, 5, 8], profit=17, nb_testes=256, nb_valides=41),
 0.00012344492882039048,
 {'reel': 0.00012358965759995044,
  'utilisateur': 0.0001182658,
  'systeme': 0.0,
  'taux_cpu': 95.69231139292955,
  'pic_memoire_mo': 67.8828125,
  'ctx_volontaires': 0.0036,
  'ctx_forces': 0.001})

### Resultat de la recherche exhaustive

Nous utiliserons ces resultats, obtenu sur mon PC portable, pour comparer avec QAOA:    

```
== Benchmark ==
Date           : 2026-11-24 22:38:23
Machine        : RulianPC (Linux 6.18.33.2-microsoft-standard-WSL2, x86_64)
Processeur     : Intel(R) Core(TM) Ultra 7 155H @ 2995 MHz
Coeurs         : 22 logiques (22 utilisables)
Charge système : 0.29 / 0.39 / 0.53 (1 / 5 / 15 min)
Mémoire        : 14.6 Go
Python         : CPython 3.12.3 (/home/julian/Quantum-Computing/.venv/bin/python)
===============
== Variables ==
10 ressources: {'Fraiseuse CNC': 'machine', 'Découpeuse laser': 'machine', 'Cabine de peinture': 'machine', 'Tour': 'machine', 'Soudeur certifié': 'ouvrier', 'Inspecteur qualité': 'ouvrier', 'Électricien': 'ouvrier', 'Créneau lundi': 'créneau', 'Créneau mercredi': 'créneau', 'Créneau vendredi': 'créneau'}
Profits des 8 commandes: [5, 3, 4, 6, 2, 5, 3, 4]
Paires des 10 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]
===============
Durée moyenne sur 10000 exécutions : 0.123 ms
256 possibilités testées par exécution
41 resultats valides (16.01562 %)
Meilleure sélection : commandes [1, 4, 5, 8], profit 17
Temps réel moyen : 0.124 ms
Temps CPU moyen  : 0.118 ms utilisateur + 0.000 ms système (96 % d'un coeur)
Pic de mémoire   : 67.9 Mo
Changements de contexte moyens : 0.0036 volontaires, 0.0010 forcés par exécution
```

Essayons maintenant avec un atelier deux fois plus grand:

### Resultat de la recherche exhaustive (x2 commandes)

In [11]:
#DEFINITION_RUN
# Atelier agrandi : on juxtapose `facteur` copies de l'atelier de base (ressources et commandes renumérotées).
def agrandir_atelier(facteur: int) -> tuple[dict[str, str], list[int], list[tuple[int, int]]]:
    ressources: dict[str, str] = {f"{nom} #{k + 1}": t for k in range(facteur) for nom, t in RESSOURCES.items()}
    commandes: list[int] = COMMANDES * facteur
    conflits: dict[str, list[int]] = {
        f"{nom} #{k + 1}": [c + k * len(COMMANDES) for c in numeros]
        for k in range(facteur)
        for nom, numeros in CONFLITS.items()
    }
    verifier_coherence(ressources, commandes, conflits)
    return ressources, commandes, calculer_paires(conflits)

# Lance et affiche le benchmark de la force brute sur l'atelier agrandi `facteur` fois.
def benchmark_atelier(facteur: int, repetitions: int) -> None:
    ressources, commandes, paires = agrandir_atelier(facteur)
    print(f"== Atelier x{facteur} : {len(commandes)} commandes, {len(paires)} paires en conflit ==")
    print_running_env()
    mesurer_avec_usage_cpu(commandes, paires, repetitions, True, ressources)

print("Defini:", agrandir_atelier, benchmark_atelier)

Defini: <function agrandir_atelier at 0x7e97a2498b80> <function benchmark_atelier at 0x7e97a2498f40>


In [12]:
benchmark_atelier(facteur=2, repetitions=10000)

== Atelier x2 : 16 commandes, 20 paires en conflit ==
== Benchmark ==
Date           : 2026-11-24 22:38:24
Machine        : RulianPC (Linux 6.18.33.2-microsoft-standard-WSL2, x86_64)
Processeur     : Intel(R) Core(TM) Ultra 7 155H @ 2995 MHz
Coeurs         : 22 logiques (22 utilisables)
Charge système : 0.35 / 0.40 / 0.53 (1 / 5 / 15 min)
Mémoire        : 14.6 Go
Python         : CPython 3.12.3 (/home/julian/Quantum-Computing/.venv/bin/python)
== Variables ==
20 ressources: {'Fraiseuse CNC #1': 'machine', 'Découpeuse laser #1': 'machine', 'Cabine de peinture #1': 'machine', 'Tour #1': 'machine', 'Soudeur certifié #1': 'ouvrier', 'Inspecteur qualité #1': 'ouvrier', 'Électricien #1': 'ouvrier', 'Créneau lundi #1': 'créneau', 'Créneau mercredi #1': 'créneau', 'Créneau vendredi #1': 'créneau', 'Fraiseuse CNC #2': 'machine', 'Découpeuse laser #2': 'machine', 'Cabine de peinture #2': 'machine', 'Tour #2': 'machine', 'Soudeur certifié #2': 'ouvrier', 'Inspecteur qualité #2': 'ouvrier', 'Élect

Durée moyenne sur 10000 exécutions : 27.877 ms
65536 possibilités testées par exécution
1681 resultats valides (2.56500 %)
Meilleure sélection : commandes [1, 4, 5, 8, 9, 12, 13, 16], profit 34
Temps réel moyen : 27.878 ms
Temps CPU moyen  : 26.668 ms utilisateur + 0.003 ms système (96 % d'un coeur)
Pic de mémoire   : 67.9 Mo
Changements de contexte moyens : 0.2313 volontaires, 0.1604 forcés par exécution


En doublant l'atelier (ressources, commandes et conflits), avec le meme nombre d'iterations, nous obtenons ces resultats:

```
== Atelier x2 : 16 commandes, 20 paires en conflit ==
== Benchmark ==
Date           : 2026-11-24 22:38:24
Machine        : RulianPC (Linux 6.18.33.2-microsoft-standard-WSL2, x86_64)
Processeur     : Intel(R) Core(TM) Ultra 7 155H @ 2995 MHz
Coeurs         : 22 logiques (22 utilisables)
Charge système : 0.35 / 0.40 / 0.53 (1 / 5 / 15 min)
Mémoire        : 14.6 Go
Python         : CPython 3.12.3 (/home/julian/Quantum-Computing/.venv/bin/python)
===============
== Variables ==
20 ressources: {'Fraiseuse CNC #1': 'machine', 'Découpeuse laser #1': 'machine', 'Cabine de peinture #1': 'machine', 'Tour #1': 'machine', 'Soudeur certifié #1': 'ouvrier', 'Inspecteur qualité #1': 'ouvrier', 'Électricien #1': 'ouvrier', 'Créneau lundi #1': 'créneau', 'Créneau mercredi #1': 'créneau', 'Créneau vendredi #1': 'créneau', 'Fraiseuse CNC #2': 'machine', 'Découpeuse laser #2': 'machine', 'Cabine de peinture #2': 'machine', 'Tour #2': 'machine', 'Soudeur certifié #2': 'ouvrier', 'Inspecteur qualité #2': 'ouvrier', 'Électricien #2': 'ouvrier', 'Créneau lundi #2': 'créneau', 'Créneau mercredi #2': 'créneau', 'Créneau vendredi #2': 'créneau'}
Profits des 16 commandes: [5, 3, 4, 6, 2, 5, 3, 4, 5, 3, 4, 6, 2, 5, 3, 4]
Paires des 20 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8), (9, 10), (9, 11), (10, 12), (11, 12), (11, 13), (12, 14), (13, 14), (13, 15), (14, 16), (15, 16)]
===============
Durée moyenne sur 10000 exécutions : 27.877 ms
65536 possibilités testées par exécution
1681 resultats valides (2.56500 %)
Meilleure sélection : commandes [1, 4, 5, 8, 9, 12, 13, 16], profit 34
Temps réel moyen : 27.878 ms
Temps CPU moyen  : 26.668 ms utilisateur + 0.003 ms système (96 % d'un coeur)
Pic de mémoire   : 67.9 Mo
Changements de contexte moyens : 0.2313 volontaires, 0.1604 forcés par exécution
```

Creeons des graphes pour afficher et comparer les resultats: 

In [13]:
#DEFINITION_RUN
# Graphes : comparaison de la recherche exhaustive sur l'atelier de base et sur l'atelier x2.
# Valeurs relevées dans les résultats ci-dessus (même machine, 10000 exécutions chacune).
import matplotlib.pyplot as plt

RESULTATS: dict[str, dict[str, float]] = {
    "Base (8 commandes)": {
        "commandes": 8, "testes": 256, "valides": 41,
        "profit": 17, "duree_ms": 0.123, "cpu_ms": 0.118 + 0.000,
    },
    "Atelier x2 (16 commandes)": {
        "commandes": 16, "testes": 65536, "valides": 1681,
        "profit": 34, "duree_ms": 27.877, "cpu_ms": 26.668 + 0.003,
    },
}
COULEURS: list[str] = ["#4C78A8", "#F58518"]

# Barres d'une seule métrique, une par atelier, avec la valeur écrite au-dessus.
def graphe_metrique(ax, resultats: dict[str, dict[str, float]], cle: str, titre: str, unite: str, log: bool = False) -> None:
    valeurs: list[float] = [r[cle] for r in resultats.values()]
    barres = ax.bar(list(resultats), valeurs, color=COULEURS[: len(valeurs)])
    ax.bar_label(barres, labels=[f"{v:g} {unite}".strip() for v in valeurs], padding=3)
    ax.set_title(titre)
    if log:
        ax.set_yscale("log")
    ax.margins(y=0.15)

# Choix examinés (2^N) contre choix valides, par atelier : montre que la part de valides s'effondre.
def graphe_espace_recherche(ax, resultats: dict[str, dict[str, float]]) -> None:
    noms: list[str] = list(resultats)
    x: list[int] = list(range(len(noms)))
    largeur: float = 0.38
    testes = ax.bar([i - largeur / 2 for i in x], [r["testes"] for r in resultats.values()], largeur, label="Testés (2^N)", color="#9AA5B1")
    valides = ax.bar([i + largeur / 2 for i in x], [r["valides"] for r in resultats.values()], largeur, label="Valides", color="#54A24B")
    ax.bar_label(testes, padding=3)
    ax.bar_label(valides, padding=3)
    ax.set_xticks(x, noms)
    ax.set_yscale("log")
    ax.set_title("Espace de recherche")
    ax.legend()
    ax.margins(y=0.15)

# Facteur de croissance de chaque métrique entre l'atelier de base et l'atelier x2.
def graphe_croissance(ax, base: dict[str, float], agrandi: dict[str, float]) -> None:
    metriques: dict[str, str] = {
        "commandes": "Commandes",
        "profit": "Profit",
        "valides": "Choix valides",
        "testes": "Choix testés",
        "duree_ms": "Durée",
    }
    facteurs: list[float] = [agrandi[c] / base[c] for c in metriques]
    barres = ax.barh(list(metriques.values()), facteurs, color="#E45756")
    ax.bar_label(barres, labels=[f"x{f:.4g}" for f in facteurs], padding=3)
    ax.set_xscale("log")
    ax.set_title("Croissance de base à x2")
    ax.margins(x=0.2)

# Figure de comparaison complète : les 4 graphes côte à côte.
def graphe_comparaison(resultats: dict[str, dict[str, float]]) -> None:
    base, agrandi = list(resultats.values())
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    graphe_metrique(axes[0, 0], resultats, "duree_ms", "Durée moyenne par exécution (échelle log)", "ms", log=True)
    graphe_espace_recherche(axes[0, 1], resultats)
    graphe_metrique(axes[1, 0], resultats, "profit", "Profit maximal trouvé", "")
    graphe_croissance(axes[1, 1], base, agrandi)
    fig.suptitle("Recherche exhaustive : atelier de base contre atelier x2", fontsize=14)
    fig.tight_layout()
    plt.show()

print("Defini:", graphe_metrique, graphe_espace_recherche, graphe_croissance, graphe_comparaison)

Defini: <function graphe_metrique at 0x7e97a1a58180> <function graphe_espace_recherche at 0x7e97a2527ce0> <function graphe_croissance at 0x7e975692e020> <function graphe_comparaison at 0x7e975692e0c0>


In [14]:
graphe_comparaison(RESULTATS)

/tmp/ipykernel_1278368/2115150408.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resolution du probleme (QAOA)

QAOA est un **algorithme hybride quantique-classique** : une partie quantique prepare et mesure un etat parametre, une partie classique ajuste ces parametres pour minimiser une energie. Il est adapte aux problemes d'optimisation combinatoire comme celui-ci.

### Modele mathematique

Soit $x_i \in \{0, 1\}$ la decision pour la commande $i$ ($x_i = 1$ si elle est acceptee), $p_i$ son profit et $E$ l'ensemble des paires de commandes en conflit. Le probleme s'ecrit:

$$\max_{x} \sum_{i} p_i x_i \quad \text{sous la contrainte} \quad x_i x_j = 0 \;\; \forall (i,j) \in E$$

Les contraintes sont ensuite retirees de l'objectif en les penalisant.

<details>
<summary>Developpement</summary>

- $\sum_i p_i x_i$ est le profit total: une commande rapporte $p_i$ si elle est acceptee ($x_i = 1$), rien sinon.
- Pour une paire $(i,j) \in E$, les deux commandes ne peuvent pas etre acceptees ensemble. Le produit $x_i x_j$ vaut $1$ seulement si $x_i = x_j = 1$, donc la contrainte $x_i x_j = 0$ exprime exactement cette regle.
- Avec $n = 8$ commandes, il y a $2^8 = 256$ selections possibles: c'est l'espace que la recherche exhaustive parcourt.

</details>

### Optimisation Binaire Quadratique sans Contrainte (Quadratic unconstrained binary optimization / QUBO)

On minimise la fonction suivante, ou $A$ est un coefficient de penalite:

$$f(x) = -\sum_{i} p_i x_i + A \sum_{(i,j) \in E} x_i x_j$$

Pour que les solutions avec conflit ne soient jamais optimales, il suffit de choisir $A > \max_i p_i$: violer une contrainte coute $A$ et ne rapporte jamais plus que le profit d'une seule commande. Ici $\max_i p_i = 6$, donc $A = 7$ convient. Le minimum de $f$ est alors une selection valide de profit maximal.

<details>
<summary>Developpement</summary>

**1. Selection valide.** Si aucune paire de $E$ n'est acceptee, tous les $x_i x_j$ sont nuls, donc $f(x) = -\sum_i p_i x_i$. Minimiser $f$ revient alors a maximiser le profit.

**2. Selection invalide.** Supposons qu'une paire $(i,j) \in E$ soit violee, et notons $k_j \geq 1$ le nombre de paires violees contenant $j$. Retirer la commande $j$ (passer $x_j$ de $1$ a $0$) change $f$ ainsi:

- le terme de profit augmente de $+p_j$ (on perd le profit de $j$);
- chaque paire violee contenant $j$ ne compte plus, ce qui retire $-A k_j$.

$$\Delta f = p_j - A\,k_j \leq p_j - A < 0 \quad \text{si } A > p_j$$

Donc toute selection invalide peut etre amelioree en retirant une commande. Le minimum de $f$ est donc atteint par une selection valide.

**3. Valeur de $A$.** Il faut $A > p_j$ pour tout $j$, soit $A > \max_i p_i$. Ici les profits sont $[5, 3, 4, 6, 2, 5, 3, 4]$, donc $\max_i p_i = 6$ et $A = 7$ est le plus petit entier qui convient.

</details>

### Hamiltonien

On remplace chaque $x_i$ par l'operateur de Pauli $Z_i$ grace a $x_i = \frac{1 - Z_i}{2}$. On obtient un Hamiltonien d'Ising:

$$H = \sum_{i} h_i Z_i + \sum_{(i,j) \in E} J_{ij} Z_i Z_j + C$$

avec $J_{ij} = \frac{A}{4}$, $h_i = \frac{p_i}{2} - \frac{A\, d_i}{4}$ ($d_i$ = nombre de conflits de la commande $i$) et $C = \frac{A\,|E|}{4} - \frac{1}{2}\sum_i p_i$.

L'etat fondamental de $H$ (energie minimale) correspond a la meilleure selection. QAOA prepare un etat parametre:

$$|\psi(\gamma, \beta)\rangle = U_B(\beta_p)\,U_C(\gamma_p)\cdots U_B(\beta_1)\,U_C(\gamma_1)\,|+\rangle^{\otimes n}, \qquad U_C(\gamma) = e^{-i\gamma H}, \quad U_B(\beta) = e^{-i\beta \sum_i X_i}$$

ou $p$ est la profondeur du circuit et $(\gamma, \beta)$ sont ses parametres.

<details>
<summary>Developpement du Hamiltonien</summary>

**1. Substitution.** Sur les etats de base, $Z_i$ vaut $+1$ si $x_i = 0$ et $-1$ si $x_i = 1$. D'ou $x_i = \frac{1 - Z_i}{2}$.

**2. Terme de profit.**

$$-\sum_i p_i \frac{1 - Z_i}{2} = -\sum_i \frac{p_i}{2} + \sum_i \frac{p_i}{2} Z_i$$

**3. Terme de penalite.** Pour chaque paire $(i,j) \in E$:

$$(1 - Z_i)(1 - Z_j) = 1 - Z_i - Z_j + Z_i Z_j$$

donc

$$A \sum_{(i,j) \in E} x_i x_j = \frac{A}{4} \sum_{(i,j) \in E} \left(1 - Z_i - Z_j + Z_i Z_j\right)$$

**4. Regrouper les termes lineaires.** Chaque commande $i$ apparait dans $d_i$ paires, donc $\sum_{(i,j) \in E} (Z_i + Z_j) = \sum_i d_i Z_i$. Il reste:

$$\frac{A}{4}\left(|E| - \sum_i d_i Z_i + \sum_{(i,j) \in E} Z_i Z_j\right)$$

**5. Identification.**

- $h_i = \frac{p_i}{2} - \frac{A\, d_i}{4}$
- $J_{ij} = \frac{A}{4}$
- $C = \frac{A\,|E|}{4} - \frac{1}{2}\sum_i p_i$

**6. Valeurs pour cet atelier** ($A = 7$, $|E| = 10$, $\sum_i p_i = 32$, donc $C = 1{,}5$). Le nombre de conflits $d_i$ se lit dans les paires $[(1,2),(1,3),(2,4),(3,4),(3,5),(4,6),(5,6),(5,7),(6,8),(7,8)]$:

| $i$ | $p_i$ | $d_i$ | $h_i = \frac{p_i}{2} - \frac{7 d_i}{4}$ |
|---|---|---|---|
| 1 | 5 | 2 | $-1{,}0$ |
| 2 | 3 | 2 | $-2{,}0$ |
| 3 | 4 | 3 | $-3{,}25$ |
| 4 | 6 | 3 | $-2{,}25$ |
| 5 | 2 | 3 | $-4{,}25$ |
| 6 | 5 | 3 | $-2{,}75$ |
| 7 | 3 | 2 | $-2{,}0$ |
| 8 | 4 | 2 | $-1{,}5$ |

Verification: $\sum_i d_i = 20 = 2\,|E|$.

**7. Pourquoi l'energie code le profit.** Pour un etat de base $|x\rangle$, $H|x\rangle = f(x)|x\rangle$ car la substitution est exacte. Le minimum de $H$ est donc le minimum de $f$, c'est-a-dire la selection valide de profit maximal.

**8. Circuit QAOA.** $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, donc $|+\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_x |x\rangle$: toutes les selections ont la meme amplitude au depart.

- $U_C(\gamma) = e^{-i\gamma H}$ est diagonal dans la base $|x\rangle$: il ajoute la phase $e^{-i\gamma f(x)}$ a chaque selection.
- $U_B(\beta) = \prod_i e^{-i\beta X_i}$ (les $X_i$ commutent), avec $e^{-i\beta X} = \cos\beta\, I - i \sin\beta\, X$. Il melange les selections en inversant des bits.

Alterner $U_C$ et $U_B$ sur $p$ couches donne l'etat $|\psi(\gamma,\beta)\rangle$.

</details>

### Boucle hybride

1. Le circuit quantique prepare $|\psi(\gamma, \beta)\rangle$, le mesure et en deduit une estimation de $\langle H \rangle$.
2. L'optimiseur classique (par exemple COBYLA) met a jour $(\gamma, \beta)$ pour faire baisser $\langle H \rangle$.
3. On repete jusqu'a convergence, puis on echantillonne le circuit final et on garde la meilleure selection **valide** trouvee.

Avec $p$ petit, la solution trouvee n'est pas forcement optimale: on la compare donc a la recherche exhaustive ci-dessus (profit 17 pour l'atelier de base, 34 pour l'atelier x2).

<details>
<summary>Developpement de la boucle hybride</summary>

**1. Valeur moyenne.** Si $\psi_x$ est l'amplitude de la selection $x$, la mesure donne $x$ avec probabilite $|\psi_x|^2$. L'energie moyenne vaut donc:

$$\langle H \rangle = \sum_x |\psi_x|^2 \, f(x)$$

**2. Estimation par echantillonnage.** Avec $N$ tirages $x^{(1)}, \dots, x^{(N)}$:

$$\langle H \rangle \approx \frac{1}{N} \sum_{s=1}^{N} f\left(x^{(s)}\right)$$

Chaque terme $Z_i$ et $Z_i Z_j$ se mesure dans la base de calcul, donc un seul type de mesure suffit.

**3. Mise a jour classique.** L'optimiseur cherche $(\gamma, \beta) = \arg\min \langle H \rangle$. Il ne voit que la valeur estimee, pas le circuit.

**4. Selection finale.** Parmi les $x^{(s)}$ mesurees, on ecarte celles qui violent une paire de $E$, puis on garde celle de profit maximal. Ce filtre est necessaire car la penalite ne garantit une solution valide qu'a l'optimum.

</details>

### Implementation

Maintenant que le modele mathematique est definie, nous pouvons l'implementer avec Qiskit et Qiskit optimization:

In [15]:
#DEFINITION_RUN
# Dependances requises : qiskit, qiskit-optimization, scipy (voir le tutoriel d'installation).
import numpy as np
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit_optimization import QuadraticProgram
from scipy.optimize import minimize

print("Imported Qiskit.")

Imported Qiskit.


#### Etape 1 : le QUBO

On decrit $f(x) = -\sum_i p_i x_i + A \sum_{(i,j) \in E} x_i x_j$ dans un `QuadraticProgram`: une variable binaire par commande, le profit en terme lineaire (negatif car on minimise) et la penalite $A$ en terme quadratique.

In [16]:
#DEFINITION_RUN
# A > max(profits) garantit qu'une selection en conflit n'est jamais optimale.
PENALITE: int = max(COMMANDES) + 1

def creer_qubo(commandes: list[int], paires: list[tuple[int, int]], penalite: int) -> QuadraticProgram:
    qp = QuadraticProgram("atelier")
    for i in range(1, len(commandes) + 1):
        qp.binary_var(f"x{i}")
    qp.minimize(
        linear={f"x{i}": -p for i, p in enumerate(commandes, start=1)},
        quadratic={(f"x{i}", f"x{j}"): penalite for i, j in paires},
    )
    return qp

print("Defini:", creer_qubo)

Defini: <function creer_qubo at 0x7e971626a200>


In [17]:
#DEFINITION_RUN
QUBO: QuadraticProgram = creer_qubo(COMMANDES, PAIRES, PENALITE)
print(QUBO.prettyprint())

Problem name: atelier

Minimize
  7*x1*x2 + 7*x1*x3 + 7*x2*x4 + 7*x3*x4 + 7*x3*x5 + 7*x4*x6 + 7*x5*x6 + 7*x5*x7
  + 7*x6*x8 + 7*x7*x8 - 5*x1 - 3*x2 - 4*x3 - 6*x4 - 2*x5 - 5*x6 - 3*x7 - 4*x8

Subject to
  No constraints

  Binary variables (8)
    x1 x2 x3 x4 x5 x6 x7 x8



#### Etape 2 : le Hamiltonien

`to_ising()` applique la substitution $x_i = \frac{1 - Z_i}{2}$ vue plus haut.    
On obtient l'operateur hamiltonien $H$ (les $h_i Z_i$ et $J_{ij} Z_i Z_j$) et la constante $C$.

In [18]:
#DEFINITION_RUN
HAMILTONIEN, CONSTANTE = QUBO.to_ising()
print(f"{HAMILTONIEN.num_qubits} qubits, {len(HAMILTONIEN)} termes, constante C = {CONSTANTE}")
print(f"Hamiltonien:\n{HAMILTONIEN}")

8 qubits, 18 termes, constante C = 1.5
Hamiltonien:
SparsePauliOp(['IIIIIIIZ', 'IIIIIIZI', 'IIIIIZII', 'IIIIZIII', 'IIIZIIII', 'IIZIIIII', 'IZIIIIII', 'ZIIIIIII', 'IIIIIIZZ', 'IIIIIZIZ', 'IIIIZIZI', 'IIIIZZII', 'IIIZIZII', 'IIZIZIII', 'IIZZIIII', 'IZIZIIII', 'ZIZIIIII', 'ZZIIIIII'],
              coeffs=[-1.  +0.j, -2.  +0.j, -3.25+0.j, -2.25+0.j, -4.25+0.j, -2.75+0.j,
 -2.  +0.j, -1.5 +0.j,  1.75+0.j,  1.75+0.j,  1.75+0.j,  1.75+0.j,
  1.75+0.j,  1.75+0.j,  1.75+0.j,  1.75+0.j,  1.75+0.j,  1.75+0.j])


#### Etape 3 : le circuit

`QAOAAnsatz` construit l'alternance $U_C(\gamma)$, $U_B(\beta)$ a partir de $H$: Hadamard sur chaque qubit (etat $|+\rangle^{\otimes n}$),    
 puis `reps` = $p$ couches. Il a $2p$ parametres: $\gamma_1, \beta_1, \dots, \gamma_p, \beta_p$.

In [19]:
#DEFINITION_RUN
PROFONDEUR: int = 3  # p : nombre de couches

In [20]:
#DEFINITION_RUN
# decompose() remplace PauliEvolutionGate par des portes RZZ/RX, beaucoup plus rapides a simuler.
CIRCUIT = QAOAAnsatz(HAMILTONIEN, reps=PROFONDEUR).decompose(reps=3)
print(f"{CIRCUIT.num_qubits} qubits, {CIRCUIT.num_parameters} parametres (gamma, beta), profondeur {CIRCUIT.depth()}, {CIRCUIT.num_nonlocal_gates()} portes a 2 qubits")
CIRCUIT.draw("mpl", fold=-1)

8 qubits, 6 parametres (gamma, beta), profondeur 46, 60 portes a 2 qubits


<Figure size 5722.22x702.333 with 1 Axes>

#### Etape 4 : la boucle hybride

- Partie quantique: `energie(parametres)` calcule $\langle H \rangle$ avec le circuit.     
- Partie classique: COBYLA ajuste $(\gamma, \beta)$ pour la faire baisser.    

On relance depuis plusieurs points de depart aleatoires, car le paysage d'energie a des minima locaux.

In [21]:
#DEFINITION_RUN
ESTIMATOR = StatevectorEstimator()  # simulation exacte, sans bruit

# Energie moyenne <H> de l'etat prepare avec ces parametres.
def energie(parametres: np.ndarray) -> float:
    pub = (CIRCUIT, HAMILTONIEN, {tuple(CIRCUIT.parameters): parametres})
    return float(ESTIMATOR.run([pub]).result()[0].data["evs"])

def optimiser(nb_departs: int, seed: int = 1):
    rng = np.random.default_rng(seed)
    essais = [
        minimize(energie, rng.uniform(0, np.pi, CIRCUIT.num_parameters), method="COBYLA", options={"maxiter": 300})
        for _ in range(nb_departs)
    ]
    return min(essais, key=lambda r: r.fun)

print("Defini:", energie, optimiser)

Defini: <function energie at 0x7e9715e1c680> <function optimiser at 0x7e9715e1c7c0>


In [22]:
#DEFINITION_RUN
debut_optimisation: float = perf_counter()
OPTIMUM = optimiser(nb_departs=3)
DUREE_OPTIMISATION: float = perf_counter() - debut_optimisation  # reprise dans la chronologie IBM
print(f"Optimisation COBYLA en {DUREE_OPTIMISATION:.1f} s")
print(f"Energie minimale <H> + C = {OPTIMUM.fun + CONSTANTE:.3f} (profit moyen moins penalites : {-(OPTIMUM.fun + CONSTANTE):.3f})")
# Qiskit trie les parametres par nom : les beta viennent avant les gamma.
print("Parametres :", {p.name: round(float(v), 3) for p, v in zip(CIRCUIT.parameters, OPTIMUM.x)})

Optimisation COBYLA en 5.0 s
Energie minimale <H> + C = -8.090 (profit moyen moins penalites : 8.090)
Parametres : {'β[0]': 2.717, 'β[1]': 1.531, 'β[2]': 1.474, 'γ[0]': 0.125, 'γ[1]': 3.643, 'γ[2]': 1.586}


#### Etape 5 : echantillonner et garder la meilleure selection valide

On mesure le circuit final 4096 fois. Qiskit ecrit les bits de droite a gauche (le qubit 0 est le dernier caractere), on inverse donc la chaine. On ecarte les selections en conflit et on garde celle de profit maximal.

In [23]:
#DEFINITION_RUN
TIRAGES: int = 4096

def echantillonner(parametres: np.ndarray, tirages: int) -> dict[str, int]:
    circuit = CIRCUIT.copy()
    circuit.measure_all()
    sampler = StatevectorSampler(seed=1)
    pub = (circuit, {tuple(circuit.parameters): parametres})
    return sampler.run([pub], shots=tirages).result()[0].data["meas"].get_counts()

# Reutilise est_valide et calculer_profit de la recherche exhaustive.
def meilleure_selection_valide(
    comptes: dict[str, int], commandes: list[int] = COMMANDES, paires: list[tuple[int, int]] = PAIRES
) -> tuple[list[int], int, float]:
    meilleur: tuple[int, ...] = (0,) * len(commandes)
    meilleur_profit: int = 0
    nb_valides: int = 0
    for bits, n in comptes.items():
        choix: tuple[int, ...] = tuple(int(b) for b in reversed(bits))
        if not est_valide(choix, paires):
            continue
        nb_valides += n
        profit: int = calculer_profit(choix, commandes)
        if profit > meilleur_profit:
            meilleur, meilleur_profit = choix, profit
    acceptees: list[int] = [i + 1 for i, x in enumerate(meilleur) if x]
    return acceptees, meilleur_profit, nb_valides / sum(comptes.values())

In [24]:
COMPTES = echantillonner(OPTIMUM.x, TIRAGES)
acceptees, profit, part_valide = meilleure_selection_valide(COMPTES)
print(f"QAOA (p={PROFONDEUR}) : commandes {acceptees}, profit {profit}")
print(f"{100 * part_valide:.1f} % des tirages sont valides")
reference: Resultat = recherche_exhaustive(COMMANDES, PAIRES)
print(f"Recherche exhaustive : commandes {reference.acceptees}, profit {reference.profit}")

QAOA (p=3) : commandes [1, 4, 5, 8], profit 17
65.7 % des tirages sont valides
Recherche exhaustive : commandes [1, 4, 5, 8], profit 17


### Benchmark de QAOA

On mesure QAOA comme la recherche exhaustive: meme machine, meme atelier (base puis x2), memes indicateurs (duree, CPU, memoire, changements de contexte).    
La mesure couvre toute la chaine: QUBO, Hamiltonien, circuit, optimisation COBYLA et echantillonnage.

QAOA est stochastique: chaque repetition utilise une graine differente et on releve **combien de fois l'optimum est trouve**.    
Il y a moins de repetitions que pour la force brute, car une execution dure plusieurs secondes.

In [25]:
#DEFINITION_RUN
# Benchmark de QAOA : meme mesure que la recherche exhaustive, sur le meme atelier.
from statistics import stdev

class ResultatQaoa(NamedTuple):
    acceptees: list[int]
    profit: int
    nb_evaluations: int  # appels a l'estimateur, tous departs confondus
    part_valide: float   # part des tirages sans conflit

# Resout le probleme de bout en bout : QUBO, Hamiltonien, circuit, COBYLA, echantillonnage.
def resoudre_qaoa(
    commandes: list[int], paires: list[tuple[int, int]], profondeur: int, nb_departs: int, tirages: int, seed: int
) -> ResultatQaoa:
    hamiltonien, _ = creer_qubo(commandes, paires, max(commandes) + 1).to_ising()
    circuit = QAOAAnsatz(hamiltonien, reps=profondeur).decompose(reps=3)
    cles = tuple(circuit.parameters)
    estimateur = StatevectorEstimator()

    def energie_locale(parametres: np.ndarray) -> float:
        pub = (circuit, hamiltonien, {cles: parametres})
        return float(estimateur.run([pub]).result()[0].data["evs"])

    rng = np.random.default_rng(seed)
    essais = [
        minimize(energie_locale, rng.uniform(0, np.pi, circuit.num_parameters), method="COBYLA", options={"maxiter": 300})
        for _ in range(nb_departs)
    ]
    optimum = min(essais, key=lambda r: r.fun)

    mesure = circuit.copy()
    mesure.measure_all()
    pub = (mesure, {tuple(mesure.parameters): optimum.x})
    comptes: dict[str, int] = StatevectorSampler(seed=seed).run([pub], shots=tirages).result()[0].data["meas"].get_counts()

    meilleur: tuple[int, ...] = (0,) * len(commandes)
    meilleur_profit: int = 0
    nb_valides: int = 0
    for bits, n in comptes.items():
        choix = tuple(int(b) for b in reversed(bits))
        if not est_valide(choix, paires):
            continue
        nb_valides += n
        profit = calculer_profit(choix, commandes)
        if profit > meilleur_profit:
            meilleur, meilleur_profit = choix, profit
    acceptees = [i + 1 for i, x in enumerate(meilleur) if x]
    return ResultatQaoa(acceptees, meilleur_profit, sum(r.nfev for r in essais), nb_valides / tirages)

# Lance QAOA `repetitions` fois (graine differente a chaque fois) et releve les memes mesures que la force brute.
def mesurer_qaoa(
    commandes: list[int], paires: list[tuple[int, int]], repetitions: int, profondeur: int, nb_departs: int, tirages: int
) -> tuple[list[ResultatQaoa], list[float], dict[str, float]]:
    assert repetitions >= 1, "repetitions doit être au moins 1"
    resultats: list[ResultatQaoa] = []
    durees: list[float] = []
    avant = resource.getrusage(resource.RUSAGE_SELF)
    debut: float = perf_counter()
    for graine in range(repetitions):
        t: float = perf_counter()
        resultats.append(resoudre_qaoa(commandes, paires, profondeur, nb_departs, tirages, graine))
        durees.append(perf_counter() - t)
    reel: float = perf_counter() - debut
    apres = resource.getrusage(resource.RUSAGE_SELF)
    utilisateur: float = apres.ru_utime - avant.ru_utime
    systeme: float = apres.ru_stime - avant.ru_stime
    usage: dict[str, float] = {
        "utilisateur": utilisateur / repetitions,
        "systeme": systeme / repetitions,
        "taux_cpu": 100 * (utilisateur + systeme) / reel,
        "pic_memoire_mo": apres.ru_maxrss / 1024,
        "ctx_volontaires": (apres.ru_nvcsw - avant.ru_nvcsw) / repetitions,
        "ctx_forces": (apres.ru_nivcsw - avant.ru_nivcsw) / repetitions,
    }
    return resultats, durees, usage

# Affiche le benchmark de QAOA sur l'atelier agrandi `facteur` fois, a comparer a `benchmark_atelier`.
def benchmark_qaoa(facteur: int, repetitions: int, profondeur: int = PROFONDEUR, nb_departs: int = 3, tirages: int = TIRAGES) -> None:
    ressources, commandes, paires = agrandir_atelier(facteur)
    optimal: int = recherche_exhaustive(commandes, paires).profit
    print(f"== QAOA atelier x{facteur} : {len(commandes)} commandes ({len(commandes)} qubits), {len(paires)} paires en conflit ==")
    print_running_env()
    print_variables_in_use(commandes, paires, ressources)
    resultats, durees, usage = mesurer_qaoa(commandes, paires, repetitions, profondeur, nb_departs, tirages)
    moyenne: float = sum(durees) / repetitions
    ecart: float = stdev(durees) if repetitions > 1 else 0.0
    trouves: int = sum(r.profit == optimal for r in resultats)
    meilleur: ResultatQaoa = max(resultats, key=lambda r: r.profit)
    print(f"Durée moyenne sur {repetitions} exécutions : {moyenne * 1000:.3f} ms (écart-type {ecart * 1000:.3f} ms)")
    print(f"p={profondeur}, {nb_departs} départs COBYLA, {tirages} tirages par exécution")
    print(f"{sum(r.nb_evaluations for r in resultats) / repetitions:.0f} évaluations de l'énergie par exécution")
    print(f"{100 * sum(r.part_valide for r in resultats) / repetitions:.1f} % des tirages valides")
    print(f"Optimum trouvé {trouves}/{repetitions} fois (profit optimal {optimal})")
    print(f"Meilleure sélection : commandes {meilleur.acceptees}, profit {meilleur.profit}")
    print(f"Temps CPU moyen  : {usage['utilisateur'] * 1000:.3f} ms utilisateur + {usage['systeme'] * 1000:.3f} ms système ({usage['taux_cpu']:.0f} % d'un coeur)")
    print(f"Pic de mémoire   : {usage['pic_memoire_mo']:.1f} Mo")
    print(f"Changements de contexte moyens : {usage['ctx_volontaires']:.4f} volontaires, {usage['ctx_forces']:.4f} forcés par exécution")

print("Defini:", resoudre_qaoa, mesurer_qaoa, benchmark_qaoa)

Defini: <function resoudre_qaoa at 0x7e9715e222a0> <function mesurer_qaoa at 0x7e9714cb93a0> <function benchmark_qaoa at 0x7e9714cb9300>


In [26]:
# QAOA sur l'atelier
benchmark_qaoa(facteur=1, repetitions=10)

== QAOA atelier x1 : 8 commandes (8 qubits), 10 paires en conflit ==
== Benchmark ==
Date           : 2026-11-24 22:42:57
Machine        : RulianPC (Linux 6.18.33.2-microsoft-standard-WSL2, x86_64)
Processeur     : Intel(R) Core(TM) Ultra 7 155H @ 2995 MHz
Coeurs         : 22 logiques (22 utilisables)
Charge système : 1.15 / 0.89 / 0.72 (1 / 5 / 15 min)
Mémoire        : 14.6 Go
Python         : CPython 3.12.3 (/home/julian/Quantum-Computing/.venv/bin/python)
== Variables ==
10 ressources: {'Fraiseuse CNC #1': 'machine', 'Découpeuse laser #1': 'machine', 'Cabine de peinture #1': 'machine', 'Tour #1': 'machine', 'Soudeur certifié #1': 'ouvrier', 'Inspecteur qualité #1': 'ouvrier', 'Électricien #1': 'ouvrier', 'Créneau lundi #1': 'créneau', 'Créneau mercredi #1': 'créneau', 'Créneau vendredi #1': 'créneau'}
Profits des 8 commandes: [5, 3, 4, 6, 2, 5, 3, 4]
Paires des 10 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]


Durée moyenne sur 10 exécutions : 4881.439 ms (écart-type 361.965 ms)
p=3, 3 départs COBYLA, 4096 tirages par exécution
871 évaluations de l'énergie par exécution
62.9 % des tirages valides
Optimum trouvé 8/10 fois (profit optimal 17)
Meilleure sélection : commandes [1, 4, 5, 8], profit 17
Temps CPU moyen  : 2808.362 ms utilisateur + 299.049 ms système (64 % d'un coeur)
Pic de mémoire   : 215.2 Mo
Changements de contexte moyens : 15973.7000 volontaires, 46.0000 forcés par exécution


In [27]:
# QAOA sur l'atelier facteur 2
benchmark_qaoa(facteur=2, repetitions=3)

== QAOA atelier x2 : 16 commandes (16 qubits), 20 paires en conflit ==
== Benchmark ==
Date           : 2026-11-24 22:43:44
Machine        : RulianPC (Linux 6.18.33.2-microsoft-standard-WSL2, x86_64)
Processeur     : Intel(R) Core(TM) Ultra 7 155H @ 2995 MHz
Coeurs         : 22 logiques (22 utilisables)
Charge système : 0.78 / 0.83 / 0.70 (1 / 5 / 15 min)
Mémoire        : 14.6 Go
Python         : CPython 3.12.3 (/home/julian/Quantum-Computing/.venv/bin/python)
== Variables ==
20 ressources: {'Fraiseuse CNC #1': 'machine', 'Découpeuse laser #1': 'machine', 'Cabine de peinture #1': 'machine', 'Tour #1': 'machine', 'Soudeur certifié #1': 'ouvrier', 'Inspecteur qualité #1': 'ouvrier', 'Électricien #1': 'ouvrier', 'Créneau lundi #1': 'créneau', 'Créneau mercredi #1': 'créneau', 'Créneau vendredi #1': 'créneau', 'Fraiseuse CNC #2': 'machine', 'Découpeuse laser #2': 'machine', 'Cabine de peinture #2': 'machine', 'Tour #2': 'machine', 'Soudeur certifié #2': 'ouvrier', 'Inspecteur qualité #2': 

Durée moyenne sur 3 exécutions : 112591.009 ms (écart-type 3332.206 ms)
p=3, 3 départs COBYLA, 4096 tirages par exécution
894 évaluations de l'énergie par exécution
47.0 % des tirages valides
Optimum trouvé 3/3 fois (profit optimal 34)
Meilleure sélection : commandes [1, 4, 5, 8, 9, 12, 13, 16], profit 34
Temps CPU moyen  : 856220.832 ms utilisateur + 731.883 ms système (761 % d'un coeur)
Pic de mémoire   : 512.4 Mo
Changements de contexte moyens : 4922.6667 volontaires, 1641.0000 forcés par exécution


Creeons maintenant des graphes pour interpreter les resultats et les comparer:

In [28]:
#DEFINITION_RUN
# Graphes : QAOA sur l'atelier de base et sur l'atelier x2.
# Valeurs relevées dans les résultats ci-dessus (même machine). Réutilise graphe_metrique et COULEURS.
RESULTATS_QAOA: dict[str, dict[str, float]] = {
    "Base (8 commandes)": {
        "commandes": 8, "tirages": 4096, "valides": round(0.629 * 4096),
        "profit": 17, "duree_ms": 4881.439, "cpu_ms": 2808.362 + 299.049,
        "evaluations": 871, "optimum_pct": 80,
    },
    "Atelier x2 (16 commandes)": {
        "commandes": 16, "tirages": 4096, "valides": round(0.470 * 4096),
        "profit": 34, "duree_ms": 112591.009, "cpu_ms": 856220.832 + 731.883,
        "evaluations": 894, "optimum_pct": 100,
    },
}

# Tirages mesurés contre tirages sans conflit, par atelier.
def graphe_tirages(ax, resultats: dict[str, dict[str, float]]) -> None:
    noms: list[str] = list(resultats)
    x: list[int] = list(range(len(noms)))
    largeur: float = 0.38
    tires = ax.bar([i - largeur / 2 for i in x], [r["tirages"] for r in resultats.values()], largeur, label="Tirages", color="#9AA5B1")
    valides = ax.bar([i + largeur / 2 for i in x], [r["valides"] for r in resultats.values()], largeur, label="Valides", color="#54A24B")
    ax.bar_label(tires, padding=3)
    ax.bar_label(valides, labels=[f"{r['valides']:g} ({100 * r['valides'] / r['tirages']:.1f} %)" for r in resultats.values()], padding=3)
    ax.set_xticks(x, noms)
    ax.set_title("Tirages valides")
    ax.legend()
    ax.margins(y=0.15)

# Facteur de croissance de chaque métrique entre l'atelier de base et l'atelier x2.
def graphe_croissance_qaoa(ax, base: dict[str, float], agrandi: dict[str, float]) -> None:
    metriques: dict[str, str] = {
        "commandes": "Commandes",
        "profit": "Profit",
        "evaluations": "Évaluations de l'énergie",
        "cpu_ms": "Temps CPU",
        "duree_ms": "Durée",
    }
    facteurs: list[float] = [agrandi[c] / base[c] for c in metriques]
    barres = ax.barh(list(metriques.values()), facteurs, color="#E45756")
    ax.bar_label(barres, labels=[f"x{f:.4g}" for f in facteurs], padding=3)
    ax.set_xscale("log")
    ax.set_title("Croissance de base à x2")
    ax.margins(x=0.2)

def graphe_qaoa(resultats: dict[str, dict[str, float]]) -> None:
    base, agrandi = list(resultats.values())
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    graphe_metrique(axes[0, 0], resultats, "duree_ms", "Durée moyenne par exécution (échelle log)", "ms", log=True)
    graphe_tirages(axes[0, 1], resultats)
    graphe_metrique(axes[1, 0], resultats, "optimum_pct", "Optimum trouvé (sur 10 puis 3 exécutions)", "%")
    graphe_croissance_qaoa(axes[1, 1], base, agrandi)
    fig.suptitle("QAOA : atelier de base contre atelier x2", fontsize=14)
    fig.tight_layout()
    plt.show()

print("Defini:", graphe_tirages, graphe_croissance_qaoa, graphe_qaoa)

Defini: <function graphe_tirages at 0x7e9714cbac00> <function graphe_croissance_qaoa at 0x7e9714cba840> <function graphe_qaoa at 0x7e9714cba340>


In [29]:
graphe_qaoa(RESULTATS_QAOA)

/tmp/ipykernel_1278368/2496621976.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
#DEFINITION_RUN
# Graphes : recherche exhaustive contre QAOA, atelier de base et atelier x2.
# Valeurs relevées dans les résultats ci-dessus (même machine).
COMPARAISON: dict[str, dict[str, dict[str, float]]] = {
    "Exhaustive": {
        "Base (8)": {"duree_ms": 0.123, "ecart_ms": 0.0, "cpu_ms": 0.118 + 0.000, "valide_pct": 100 * 41 / 256, "optimum_pct": 100},
        "x2 (16)": {"duree_ms": 27.877, "ecart_ms": 0.0, "cpu_ms": 26.668 + 0.003, "valide_pct": 100 * 1681 / 65536, "optimum_pct": 100},
    },
    "QAOA": {
        "Base (8)": {"duree_ms": 4881.439, "ecart_ms": 361.965, "cpu_ms": 2808.362 + 299.049, "valide_pct": 62.9, "optimum_pct": 80},
        "x2 (16)": {"duree_ms": 112591.009, "ecart_ms": 3332.206, "cpu_ms": 856220.832 + 731.883, "valide_pct": 47.0, "optimum_pct": 100},
    },
}
COULEURS_ALGO: dict[str, str] = {"Exhaustive": "#4C78A8", "QAOA": "#F58518"}

# Barres groupées : un groupe par atelier, une barre par algorithme.
def barres_groupees(ax, cle: str, titre: str, unite: str, log: bool = False, erreur: str | None = None) -> None:
    ateliers: list[str] = list(next(iter(COMPARAISON.values())))
    largeur: float = 0.38
    for k, (algo, donnees) in enumerate(COMPARAISON.items()):
        x = [i + (k - 0.5) * largeur for i in range(len(ateliers))]
        valeurs: list[float] = [donnees[a][cle] for a in ateliers]
        yerr = [donnees[a][erreur] for a in ateliers] if erreur else None
        barres = ax.bar(x, valeurs, largeur, label=algo, color=COULEURS_ALGO[algo], yerr=yerr, capsize=4)
        ax.bar_label(barres, labels=[(f"{v:,.0f}" if v >= 1000 else f"{v:.4g}") + f" {unite}".rstrip() for v in valeurs], padding=3, fontsize=8)
    ax.set_xticks(range(len(ateliers)), ateliers)
    ax.set_title(titre)
    ax.legend()
    if log:
        ax.set_yscale("log")
    ax.margins(y=0.2)

# Facteur de croissance base -> x2 : QAOA grossit bien moins vite que la force brute (2^N).
def graphe_croissance_algos(ax) -> None:
    metriques: dict[str, str] = {"duree_ms": "Durée", "cpu_ms": "Temps CPU"}
    x = range(len(metriques))
    largeur: float = 0.38
    for k, (algo, donnees) in enumerate(COMPARAISON.items()):
        base, agrandi = list(donnees.values())
        facteurs: list[float] = [agrandi[m] / base[m] for m in metriques]
        barres = ax.bar([i + (k - 0.5) * largeur for i in x], facteurs, largeur, label=algo, color=COULEURS_ALGO[algo])
        ax.bar_label(barres, labels=[f"x{f:.3g}" for f in facteurs], padding=3)
    ax.set_xticks(list(x), list(metriques.values()))
    ax.set_yscale("log")
    ax.set_title("Croissance de base à x2 (échelle log)")
    ax.legend()
    ax.margins(y=0.2)

def graphe_comparaison_qaoa() -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    barres_groupees(axes[0, 0], "duree_ms", "Durée moyenne par exécution (échelle log)", "ms", log=True, erreur="ecart_ms")
    graphe_croissance_algos(axes[0, 1])
    barres_groupees(axes[1, 0], "valide_pct", "Part de choix valides (exhaustive) / tirages valides (QAOA)", "%")
    barres_groupees(axes[1, 1], "optimum_pct", "Optimum trouvé", "%")
    axes[1, 1].set_ylim(0, 130)
    fig.suptitle("Recherche exhaustive contre QAOA : atelier de base et atelier x2", fontsize=14)
    fig.tight_layout()
    plt.show()

print("Defini:", barres_groupees, graphe_croissance_algos, graphe_comparaison_qaoa)

Defini: <function barres_groupees at 0x7e9714c509a0> <function graphe_croissance_algos at 0x7e9714c50ae0> <function graphe_comparaison_qaoa at 0x7e9714c51580>


In [31]:
graphe_comparaison_qaoa()

/tmp/ipykernel_1278368/4204396346.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation des resultats

### Algorithme classique contre Algorithme mixte

- Les resultats optimum sont similaires: les 2 algorithmes trouvent les commandes [1, 4, 5, 8] (profit 17) sur l'atelier de base et le profit 34 sur l'atelier x2.
- L'algo exhaustive est deterministe et **garantit** l'optimum. QAOA est stochastique: il l'a trouve 8 fois sur 10 (base) et 3 fois sur 3 (x2), et il faut verifier ses tirages contre les conflits.
- Seule une minorite des tirages de QAOA est valide (62.9 % puis 47.0 %), mais c'est bien plus que la part de selections valides dans l'espace de recherche (16.0 % puis 2.6 %): le circuit concentre bien la mesure sur les bonnes selections.
- Sur notre machine, l'algo exhaustive est **bien plus rapide**: 0.12 ms contre ~4.9 s sur 8 commandes (~40 000 fois), 28 ms contre 113 s sur 16 commandes (~4 000 fois).

### Affirmations basees sur notre exemple d'atelier

- A 8 et 16 commandes, la force brute reste imbattable: $2^{16} = 65\,536$ candidats se testent en quelques millisecondes.
- Le cout de la force brute explose avec $n$ (x227 en doublant les commandes, proche des $2^8 = 256$ attendus), alors que celui de QAOA augmente beaucoup moins vite (x23). Le nombre d'evaluations de l'energie reste presque constant (871 puis 894).
- Ce resultat est a nuancer: QAOA est ici **simule** sur un processeur classique. Le cout de la simulation double a chaque qubit ajoute (761 % d'un coeur a 16 qubits, temps CPU x276), et COBYLA est plafonne a 300 iterations. On ne peut donc pas conclure a un avantage quantique.
- Pour que QAOA devienne interessant, il faudrait un materiel quantique reel et un atelier bien plus grand, ou la force brute n'est plus praticable. Il faudrait aussi verifier que la qualite des solutions tient quand $n$ augmente, avec une profondeur $p$ plus grande.


## Conclusion (QAOA sur emulateur)

Choisir les commandes d'un atelier sans conflit de ressources est un probleme MWIS, que l'on peut ecrire comme un QUBO puis resoudre avec QAOA.

Sur notre exemple, QAOA retrouve le meme profit optimal que la recherche exhaustive (17 sur 8 commandes, 34 sur 16), mais sans le garantir: 8 fois sur 10 sur l'atelier de base.
La force brute reste bien plus rapide, car nos ateliers sont petits et QAOA est simule sur un processeur classique.

Cependant, l'emulateur ajoute un overhead qui nous empeche de voir des resultats reels, alors executons ce programme sur une vraie machine quantique:


## Execution sur une machine IBM

> **Correctif (job bloque en RUNNING).** Sur l'atelier x2, le circuit contient deux groupes de 8 qubits independants. Envoye d'un seul bloc, il restait bloque en RUNNING cote IBM, avant meme l'execution, sur ibm_fez, ibm_marrakesh et ibm_kingston (cause exacte non identifiee: ni les tirages, ni le materiel, ni la barriere seule). Envoye comme un PUB par groupe, il s'execute en quelques secondes. On separe donc le circuit transpile par groupe de qubits, puis on recolle les tirages; un delai maximal annule le job s'il reste bloque.

Pour juger QAOA sur un vrai materiel, on reutilise les angles $(\gamma, \beta)$ trouves par la simulation (`OPTIMUM.x`) et on ne fait qu'**echantillonner** le circuit sur une machine IBM. Cela evite de payer l'optimisation COBYLA en temps quantique.

- Le circuit est **transpile** vers les portes natives et la topologie de la machine choisie: sa profondeur reelle est plus grande que celle du circuit ideal.
- Le bruit de la machine fait baisser la part de tirages valides: c'est ce que l'on compare avec la simulation.
- Il faut un compte IBM Quantum enregistre une fois avec `QiskitRuntimeService.save_account(...)`, puis `EXECUTER_SUR_IBM = True`.
- Les resultats sont affiches directement a la fin du job (rien n'est ecrit sur disque): relancer la cellule consomme du temps machine.

<details>
<summary><b>Tutoriel : se connecter a IBM Quantum</b></summary>

#### 1. Creer un compte
- Aller sur [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) et se connecter avec un IBMid.
- Choisir un plan. Le plan **Open** est gratuit et inclut un temps machine mensuel limite, suffisant pour un job de 8 qubits.

#### 2. Recuperer les identifiants
Depuis le tableau de bord, copier:
- la **cle API** (affichee une seule fois, a conserver);
- le **CRN de l'instance** (page *Instances*).

La cle API est un mot de passe: ne jamais la committer ni l'ecrire dans le notebook.

#### 3. Preparer l'environnement
```bash
python3 -m venv .venv
source .venv/bin/activate   # Windows : .venv\\Scripts\\activate
pip install qiskit qiskit-optimization qiskit-aer qiskit-ibm-runtime scipy networkx matplotlib pylatexenc jupyterlab
```
Python 3.10 ou plus recent est necessaire. Lancer ensuite Jupyter depuis ce meme environnement (`jupyter lab`).

#### 4. Enregistrer le compte (une seule fois, dans un terminal)
```bash
python - <<'EOF'
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(
    token="VOTRE_CLE_API",
    instance="VOTRE_CRN",
    channel="ibm_quantum_platform",
    overwrite=True,
)
EOF
```
Les identifiants sont ecrits dans `~/.qiskit/qiskit-ibm.json`, hors du depot. La cellule suivante les lit automatiquement.

#### 5. Tester la connexion
```bash
python -c "
from qiskit_ibm_runtime import QiskitRuntimeService
for b in QiskitRuntimeService().backends(simulator=False, operational=True):
    print(b.name, b.num_qubits, b.status().pending_jobs, 'jobs en attente')
"
```
Toute machine d'au moins 8 qubits convient.

#### 6. Lancer la cellule
Passer `EXECUTER_SUR_IBM` a `True` et executer la cellule ci-dessous. Elle attend la fin du job (la file d'attente peut durer de quelques minutes a plus). Le job est aussi visible dans *Workloads* sur le tableau de bord IBM.

#### 7. Apres l'execution
Repasser `EXECUTER_SUR_IBM` a `False` pour qu'un "tout executer" ne consomme pas de temps machine.

| Probleme | Cause probable |
|---|---|
| `Unable to find account` | `save_account` non execute, ou execute avec un autre utilisateur |
| Erreur d'authentification | Mauvaise cle, `channel` ou `instance` |
| Aucune machine listee | Le plan n'y donne pas acces, ou maintenance |
| Job longtemps en attente | Normal avec le plan Open: recuperer le resultat plus tard avec `service.job("ID_DU_JOB").result()` |

</details>

In [32]:
# Teste la connexion au service IBM en affichant les infos des machines disponibles 
from qiskit_ibm_runtime import QiskitRuntimeService
from datetime import datetime

print(datetime.now())
for b in QiskitRuntimeService().backends(simulator=False, operational=True):
    print(b.name, b.num_qubits, b.status().pending_jobs, 'jobs en attente')

2026-11-24 22:49:07.575197


ibm_fez 156 4 jobs en attente


ibm_kingston 156 1 jobs en attente


ibm_marrakesh 156 0 jobs en attente


In [33]:
#DEFINITION_RUN
# Execution de QAOA sur IBM Cloud : tout le code est dans des fonctions, comme `benchmark_qaoa`.
from collections import Counter
from datetime import datetime
from time import perf_counter, sleep
from typing import Any, Callable, cast

import qiskit
from qiskit import ClassicalRegister, QuantumCircuit
import qiskit_ibm_runtime
from qiskit.primitives.containers import PrimitiveResult, SamplerPubResult
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

def _mediane(valeurs) -> float | None:
    v = [x for x in valeurs if x is not None]
    return float(np.median(v)) if v else None

def _essayer(fonction: Callable[[], Any]) -> Any:
    # Les metadonnees varient selon les versions de qiskit-ibm-runtime : ne jamais faire echouer le job.
    try:
        return fonction()
    except Exception:
        return None

# Affiche le temps ecoule depuis `depart` a la fin de chaque etape.
def _journal(depart: float, etape: str) -> None:
    print(f"[{perf_counter() - depart:7.1f} s] {etape}")

def _instant(texte: str) -> datetime:
    return datetime.fromisoformat(texte.replace("Z", "+00:00"))

# Proprietes de la machine et du circuit transpile (bruit, portes, qubits physiques).
def infos_machine(machine, circuit) -> dict:
    cible = machine.target
    porte_2q = next((n for n in ("cz", "ecr", "cx") if n in cible.operation_names), None)
    fidelite = 1.0
    for inst in circuit.data:  # produit des (1 - erreur) de chaque porte et mesure
        nom = inst.operation.name
        if nom in cible.operation_names:
            props = cible[nom].get(tuple(circuit.find_bit(q).index for q in inst.qubits))
            if props is not None and props.error:
                fidelite *= 1 - props.error
    return {
        "t1_s": _mediane(q.t1 for q in cible.qubit_properties),
        "t2_s": _mediane(q.t2 for q in cible.qubit_properties),
        "erreur_lecture": _mediane(p.error for p in cible["measure"].values() if p),
        "porte_2q": porte_2q,
        "erreur_2q": _mediane(p.error for p in cible[porte_2q].values() if p) if porte_2q else None,
        "derniere_calibration": _essayer(lambda: machine.properties().last_update_date),
        "portes_base": sorted(set(cible.operation_names) - {"measure", "delay", "reset"}),
        "clops": _essayer(lambda: machine.configuration().clops),
        "version_backend": str(machine.backend_version),
        "portes": dict(circuit.count_ops()),
        "portes_2q": circuit.num_nonlocal_gates(),
        "qubits_physiques": _essayer(lambda: sorted(circuit.layout.final_index_layout())),
        "fidelite_estimee": fidelite,
    }

# Duree de chaque etape d'une execution IBM (None si non mesuree). Les durees d'attente et d'execution
# viennent des horodatages d'IBM, les autres sont mesurees ici.
def chronologie_ibm(sauvegarde: dict) -> dict[str, float | None]:
    c: dict = sauvegarde.get("chrono", {})
    t: dict = sauvegarde.get("horodatages", {})
    attente = (_instant(t["running"]) - _instant(t["created"])).total_seconds() if t else None
    execution = (_instant(t["finished"]) - _instant(t["running"])).total_seconds() if t else None
    mesures = [c.get("optimisation_s"), c.get("transpilation_s"), c.get("soumission_resultat_s")]
    return {
        "optimisation_s": c.get("optimisation_s"),
        "transpilation_s": c.get("transpilation_s"),
        "attente_s": attente,
        "execution_s": execution,
        "soumission_resultat_s": c.get("soumission_resultat_s"),
        "total_s": sum(m for m in mesures if m is not None) if None not in mesures else None,
    }

# Un sous-circuit par groupe de qubits relies par des portes, avec les indices des bits classiques d'origine.
def separer_par_groupe(circuit) -> list[tuple[Any, list[int]]]:
    parent: list[int] = list(range(circuit.num_qubits))

    def racine(q: int) -> int:
        while parent[q] != q:
            q = parent[q]
        return q

    for inst in circuit.data:
        if inst.operation.name != "barrier":
            indices = [circuit.find_bit(q).index for q in inst.qubits]
            for q in indices[1:]:
                parent[racine(q)] = racine(indices[0])
    groupes: dict[int, list] = {}
    for inst in circuit.data:
        if inst.operation.name == "barrier":
            continue
        operations, bits = groupes.setdefault(racine(circuit.find_bit(inst.qubits[0]).index), [[], []])
        if inst.operation.name == "measure":
            bits.append(circuit.find_bit(inst.clbits[0]).index)
        operations.append(inst)
    resultat: list[tuple[Any, list[int]]] = []
    for operations, bits in groupes.values():
        if not bits:
            continue  # groupe sans mesure : n'influence aucun tirage
        sous = QuantumCircuit(circuit.qubits, ClassicalRegister(len(bits), "meas"))
        k = 0
        for inst in operations:
            if inst.operation.name == "measure":
                sous.measure(inst.qubits[0], sous.clbits[k])
                k += 1
            else:
                sous.append(inst.operation, inst.qubits)
        resultat.append((sous, bits))
    return resultat

# Recolle les tirages des groupes : ils sont independants, donc le i-eme tirage de chacun forme un tirage complet.
def recoller_tirages(resultat, groupes: list[tuple[Any, list[int]]], nb_bits: int) -> dict[str, int]:
    comptes: Counter[str] = Counter()
    for tirage in zip(*(pub.data["meas"].get_bitstrings() for pub in resultat)):
        bits = ["0"] * nb_bits
        for (_, indices), chaine in zip(groupes, tirage):
            for k, indice in enumerate(indices):
                bits[indice] = chaine[-1 - k]  # le bit 0 est le dernier caractere
        comptes["".join(reversed(bits))] += 1
    return dict(comptes)

# Transpile, soumet le job et retourne un dict avec les comptes et les mesures faites par IBM.
def executer_sur_ibm(
    circuit, parametres: np.ndarray, tirages: int, machine_ibm: str | None = None, delai_max_s: float = 180
) -> dict:
    depart: float = perf_counter()
    service = QiskitRuntimeService()
    if machine_ibm:
        machine = service.backend(machine_ibm)
    else:
        machine = service.least_busy(operational=True, simulator=False, min_num_qubits=circuit.num_qubits)
    _journal(depart, f"machine choisie : {machine.name}")
    mesure = circuit.copy()
    mesure.measure_all()
    debut: float = perf_counter()
    circuit_ibm = generate_preset_pass_manager(optimization_level=3, backend=machine, seed_transpiler=1).run(mesure)
    transpilation_s: float = perf_counter() - debut
    _journal(depart, f"transpilation terminee : profondeur {circuit_ibm.depth()} (circuit ideal : {mesure.depth()})")

    debut = perf_counter()
    # Un PUB par groupe de qubits : envoye d'un seul bloc, le circuit x2 reste bloque en RUNNING cote IBM.
    groupes = separer_par_groupe(circuit_ibm)
    valeurs = dict(zip(circuit.parameters, parametres))
    pubs = [(sous, [valeurs[p] for p in sous.parameters]) for sous, _ in groupes]
    job = SamplerV2(mode=machine).run(pubs, shots=tirages)
    _journal(depart, f"job soumis : {job.job_id()} ({len(pubs)} PUB)")
    statut: str | None = None
    debut_execution: float | None = None
    while True:  # sonde le statut pour voir ou en est le job (file d'attente, execution)
        courant = str(job.status())
        if courant != statut:
            statut = courant
            _journal(depart, f"statut du job : {statut}")
        if statut in ("DONE", "ERROR", "CANCELLED"):
            break
        if statut == "RUNNING" and debut_execution is None:
            debut_execution = perf_counter()
        if debut_execution is not None and perf_counter() - debut_execution > delai_max_s:
            job.cancel()
            raise TimeoutError(f"job {job.job_id()} en RUNNING depuis plus de {delai_max_s} s : annule")
        sleep(2)
    # SamplerV2 renvoie un PrimitiveResult ; le cast ne sert qu'au verificateur de types.
    resultat = cast("PrimitiveResult[SamplerPubResult]", job.result())
    soumission_resultat_s: float = perf_counter() - debut  # soumission + file d'attente + execution
    _journal(depart, "resultat recupere")
    metriques = job.metrics()  # mesures faites par IBM, pas par cette machine
    spans = _essayer(lambda: resultat.metadata["execution"]["execution_spans"])
    extra = infos_machine(machine, circuit_ibm) | {
        "statut_job": str(job.status()),
        "primitive": job.primitive_id,
        "instance": (_essayer(service.active_account) or {}).get("instance"),
        "duree_spans_s": _essayer(lambda: (spans.stop - spans.start).total_seconds()),
        "version_qiskit": qiskit.__version__,
        "version_runtime": qiskit_ibm_runtime.__version__,
    }
    _journal(depart, "metriques IBM et proprietes de la machine lues")
    return {
        "machine": machine.name, "job_id": job.job_id(), "tirages": tirages,
        "profondeur": circuit_ibm.depth(), "parametres": list(map(float, parametres)),
        "comptes": recoller_tirages(resultat, groupes, circuit_ibm.num_clbits),
        "qubits_machine": machine.num_qubits, "jobs_en_attente": machine.status().pending_jobs,
        "processeur": str(getattr(machine, "processor_type", None)),
        "horodatages": metriques.get("timestamps", {}), "usage": metriques.get("usage", {}),
        "extra": extra,
        "chrono": {"transpilation_s": transpilation_s, "soumission_resultat_s": soumission_resultat_s},
    }

# Affiche la machine IBM, les temps mesures par IBM et la meilleure selection valide.
def afficher_resultats_ibm(
    sauvegarde: dict, commandes: list[int], paires: list[tuple[int, int]], ressources: dict[str, str] | None = None
) -> None:
    acceptees, profit, part_valide = meilleure_selection_valide(sauvegarde["comptes"], commandes, paires)
    print_variables_in_use(commandes, paires, ressources)
    t: dict = sauvegarde.get("horodatages", {})
    if t:
        print("== Machine quantique IBM ==")
        print(f"Machine        : {sauvegarde['machine']} (IBM Quantum, cloud)")
        print(f"Processeur     : {sauvegarde['processeur']}")
        print(f"Qubits         : {sauvegarde['qubits_machine']}")
        print(f"File d'attente : {sauvegarde['jobs_en_attente']} jobs en attente à la soumission")
        print(f"Job            : {sauvegarde['job_id']}")
        print(f"Date           : {t.get('created')}")
        attente: float = (_instant(t["running"]) - _instant(t["created"])).total_seconds()
        print(f"Attente reelle : {attente:.1f} s (creation -> debut d'execution)")
        quantique = sauvegarde.get("usage", {}).get("quantum_seconds")
        if quantique is not None:
            print(f"Temps facture  : {quantique} s de temps quantique")
        x: dict = sauvegarde.get("extra", {})  # absent des anciens fichiers
        if x:
            print(f"Statut du job  : {x.get('statut_job')} (primitive {x.get('primitive')}, instance {x.get('instance')})")
            print(f"Backend        : version {x.get('version_backend')}, CLOPS {x.get('clops')}")
            print(f"Portes de base : {', '.join(x.get('portes_base', []))}")
            print(f"Calibration    : {x.get('derniere_calibration')}")
            if x.get("t1_s") is not None and x.get("t2_s") is not None:
                print(f"T1 / T2 median : {x['t1_s'] * 1e6:.1f} / {x['t2_s'] * 1e6:.1f} us")
            if x.get("erreur_lecture") is not None:
                print(f"Erreur lecture : {100 * x['erreur_lecture']:.2f} % (mediane)")
            if x.get("erreur_2q") is not None:
                print(f"Erreur {x['porte_2q']} : {100 * x['erreur_2q']:.2f} % (mediane, porte a 2 qubits)")
            print(f"Portes         : {x.get('portes')} dont {x.get('portes_2q')} a 2 qubits")
            print(f"Qubits utilises: {len(x.get('qubits_physiques') or [])} {x.get('qubits_physiques')}")
            print(f"Fidelite estimee du circuit : {100 * x['fidelite_estimee']:.2f} %")
            print(f"Qiskit {x.get('version_qiskit')}, qiskit-ibm-runtime {x.get('version_runtime')}")
    ch = chronologie_ibm(sauvegarde)
    print("== Chronologie ==")
    for nom, cle in (
        ("Optimisation (emulateur, local)", "optimisation_s"), ("Transpilation (local)", "transpilation_s"),
        ("File d'attente IBM", "attente_s"), ("Execution IBM", "execution_s"),
        ("Soumission -> resultat (local)", "soumission_resultat_s"), ("Total de bout en bout", "total_s"),
    ):
        print(f"{nom:32}: {'non mesuree' if ch[cle] is None else f'{ch[cle]:.1f} s'}")
    print("== Resultats ==")
    print(f"commandes {acceptees}, profit {profit}")
    print(f"{100 * part_valide:.1f} % des tirages valides")
    if t:
        execution: float = (_instant(t["finished"]) - _instant(t["running"])).total_seconds()
        print(f"{sauvegarde['tirages']} tirages par exécution")
        print(f"Temps d'exécution sur IBM : {execution * 1000:.3f} ms")
        spans_s = sauvegarde.get("extra", {}).get("duree_spans_s")
        if spans_s is not None:
            print(f"Temps sur le processeur (execution spans) : {spans_s * 1000:.3f} ms")

# Equivalent IBM de `benchmark_qaoa` : un seul appel pour executer et afficher.
# Desactive par defaut (`executer=False`) car cela consomme du temps machine.
def benchmark_qaoa_ibm(
    circuit, parametres: np.ndarray, commandes: list[int], paires: list[tuple[int, int]],
    tirages: int = TIRAGES, machine_ibm: str | None = None, executer: bool = True,
    duree_optimisation_s: float | None = None, ressources: dict[str, str] | None = None,
) -> dict | None:
    if not executer:
        print("Pas encore de resultats IBM : passer executer a True pour les obtenir.")
        return None
    sauvegarde = executer_sur_ibm(circuit, parametres, tirages, machine_ibm)
    if duree_optimisation_s is not None:
        sauvegarde["chrono"]["optimisation_s"] = duree_optimisation_s
    afficher_resultats_ibm(sauvegarde, commandes, paires, ressources)
    return sauvegarde

print("Defini:", executer_sur_ibm, afficher_resultats_ibm, benchmark_qaoa_ibm)


Defini: <function executer_sur_ibm at 0x7e9756c9ede0> <function afficher_resultats_ibm at 0x7e97568edee0> <function benchmark_qaoa_ibm at 0x7e9714a532e0>


In [34]:
# Execution sur une machine IBM (suivez le tutoriel pour connecter l'environement a un compte IBM Cloud).
EXECUTER_SUR_IBM: bool = True
MACHINE_IBM: str | None = "ibm_fez"  # ex. "ibm_marrakesh" ; None = la moins chargee

sauvegarde = benchmark_qaoa_ibm(
    CIRCUIT, OPTIMUM.x, COMMANDES, PAIRES, tirages=TIRAGES, machine_ibm=MACHINE_IBM, executer=EXECUTER_SUR_IBM,
    duree_optimisation_s=DUREE_OPTIMISATION,
)


[    3.3 s] machine choisie : ibm_fez


[    4.5 s] transpilation terminee : profondeur 278 (circuit ideal : 47)


[    6.7 s] job soumis : dc30cots2omc738cnhn0 (1 PUB)


[    7.4 s] statut du job : QUEUED


[    9.7 s] statut du job : RUNNING


[   14.3 s] statut du job : DONE


[   14.6 s] resultat recupere


[   15.1 s] metriques IBM et proprietes de la machine lues


== Variables ==
10 ressources: {'Fraiseuse CNC': 'machine', 'Découpeuse laser': 'machine', 'Cabine de peinture': 'machine', 'Tour': 'machine', 'Soudeur certifié': 'ouvrier', 'Inspecteur qualité': 'ouvrier', 'Électricien': 'ouvrier', 'Créneau lundi': 'créneau', 'Créneau mercredi': 'créneau', 'Créneau vendredi': 'créneau'}
Profits des 8 commandes: [5, 3, 4, 6, 2, 5, 3, 4]
Paires des 10 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]
== Machine quantique IBM ==
Machine        : ibm_fez (IBM Quantum, cloud)
Processeur     : {'family': 'Heron', 'revision': '2'}
Qubits         : 156
File d'attente : 4 jobs en attente à la soumission
Job            : dc30cots2omc738cnhn0
Date           : 2026-11-24T21:49:23.422977Z
Attente reelle : 2.1 s (creation -> debut d'execution)
Statut du job  : DONE (primitive executor, instance crn:v1:bluemix:public:quantum-computing:us-east:a/61e658cacb2547a68a8e83187c424ecd:76c49251-1bf7-4d63-b608-a6a4a057cf10::)
Backend  

Pour comparaison, une execution precedente de l'atelier de base sur ibm_marrakesh (Open Instance de IBM Cloud):

```
ibm_marrakesh : profondeur 281 apres transpilation (circuit ideal : 47)
Job soumis : dc2t1rds2omc738cimbg
== Variables ==
10 ressources: {'Fraiseuse CNC': 'machine', 'Découpeuse laser': 'machine', 'Cabine de peinture': 'machine', 'Tour': 'machine', 'Soudeur certifié': 'ouvrier', 'Inspecteur qualité': 'ouvrier', 'Électricien': 'ouvrier', 'Créneau lundi': 'créneau', 'Créneau mercredi': 'créneau', 'Créneau vendredi': 'créneau'}
Profits des 8 commandes: [5, 3, 4, 6, 2, 5, 3, 4]
Paires des 10 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]
===============
== Machine quantique IBM ==
Machine        : ibm_marrakesh (IBM Quantum, cloud)
Processeur     : {'family': 'Heron', 'revision': '2'}
Qubits         : 156
File d'attente : 0 jobs en attente à la soumission
Job            : dc2t1rds2omc738cimbg
Date           : 2026-11-24T18:01:17.613676Z
Attente reelle : 1.1 s (creation -> debut d'execution)
Statut du job  : DONE (primitive executor, instance crn:v1:bluemix:public:quantum-computing:us-east:a/61e658cacb2547a68a8e83187c424ecd:76c49251-1bf7-4d63-b608-a6a4a057cf10::)
Backend        : version 1.0.21, CLOPS None
Portes de base : cz, id, if_else, measure_2, measure_reset, measure_reset_2, reset_2, rz, sx, x, xslow
Calibration    : 2026-11-24 18:38:25+01:00
T1 / T2 median : 169.0 / 79.5 us
Erreur lecture : 1.98 % (mediane)
Erreur cz : 0.25 % (mediane, porte a 2 qubits)
Portes         : {'rz': 245, 'sx': 239, 'cz': 108, 'x': 9, 'measure': 8, 'barrier': 1} dont 108 a 2 qubits
Qubits utilises: 8 [32, 33, 34, 39, 52, 53, 54, 55]
Fidelite estimee du circuit : 68.52 %
Qiskit 2.5.2, qiskit-ibm-runtime 0.52.1
== Resultats ==
commandes [1, 4, 5, 8], profit 17
47.8 % des tirages valides
4096 tirages par exécution
Temps d'exécution sur IBM : 4708.777 ms
```

QAOA sur une vraie machine retourne la meme solution optimale que sur emulateur. 

<details>
<summary><b>Note de debugage : job x2 bloque en RUNNING sur IBM Cloud (a lire avant de modifier cette section)</b></summary>

#### Symptome
- Atelier x2 (16 qubits, deux copies independantes de l'atelier de base): le job passe de `QUEUED` a `RUNNING` en ~2 s, puis reste en `RUNNING` indefiniment (plus de 2 h observees), sans horodatage `finished`, sans usage et sans logs.
- Le meme code sur l'atelier de base (8 qubits) termine en ~5 s, sur le meme compte et la meme machine.
- Environnement: Python 3.12, Qiskit 2.5.2, qiskit-ibm-runtime 0.52.1, plan Open, `SamplerV2(mode=machine)` (mode job, sans session ni batch).

#### Ce qui a ete verifie
- Apres annulation, `job.metrics()["circuits_execution_time_ns"]` vaut `0`: le circuit n'a **jamais ete execute**. Le blocage a lieu cote IBM, avant l'execution sur le QPU.
- Dans qiskit-ibm-runtime 0.52.1, `SamplerV2` est le nouveau Sampler cote client, construit sur le programme `executor` (d'ou `primitive_id = "executor"`). L'ancien Sampler (`qiskit_ibm_runtime.server_side.SamplerV2`, programme `sampler`) bloque de la meme facon: ce n'est pas propre a `executor`.
- Qubits sains (aucun qubit ni porte marque defectueux), duree du circuit ~7 us, angles finis et identiques a ceux de l'atelier de base.
- Le circuit x2 transpile forme bien **deux groupes de 8 qubits sans aucune porte entre eux**.
- `job.logs()` reste vide, meme pour un job termine: il n'apporte rien ici.

#### Tests effectues (job annule automatiquement apres 60 s en `RUNNING`)

| Variante soumise | Machine | Job(s) | Resultat |
|---|---|---|---|
| x2 d'un seul bloc (code d'origine) | ibm_fez, ibm_marrakesh | `dc2tkpthdvas73bmrpf0`, `dc2tugts2omc738cjqag` | bloque |
| x2, ancien Sampler (`sampler`) | ibm_fez | `dc2up7thdvas73bmt4mg` | bloque |
| x2, `optimization_level=1` | ibm_kingston | `dc2upvts2omc738ckqu0` | bloque |
| x2 coupe en 2 PUB (un par groupe) | ibm_fez | `dc2upouq4uuc73a607gg` | termine (7.5 s) |
| base + 8 qubits inactifs mesures (16 bits) | ibm_fez | `dc2ups5s2omc738ckqq0` | termine |
| x2 sans barriere (disposition A) | ibm_fez | `dc2urr5hdvas73bmt85g`, `dc2v4r5hdvas73bmtmdg` | termine (2 fois) |
| x2 sans barriere (disposition A), 1 tirage | ibm_fez | `dc2v5l5hdvas73bmtnfg` | termine |
| x2 avec une barriere par groupe (disposition A) | ibm_fez | `dc2usfds2omc738cku90` | termine |
| x2 d'origine (temoin, disposition A) | ibm_fez | `dc2ushls2omc738ckuc0` | bloque |
| x2 sans barriere (disposition B, re-transpile) | ibm_fez | `dc2uvu2nj2vs73dhcotg`, `dc2v5n5s2omc738clcng` | bloque (1 et 4096 tirages) |
| x2 disposition B coupe en 2 PUB | ibm_fez | `dc2v4tqnj2vs73dhd2bg` | termine (13.5 s) |
| **Version finale du notebook** (2 PUB) | ibm_fez | `dc2vcbthdvas73bmu1cg` | **termine (11 s)** |

La "disposition" designe les qubits physiques choisis par la transpilation: elle change quand la calibration de la machine change.

#### Cause connue
- **Declencheur**: soumettre d'un seul bloc un circuit qui contient deux groupes de qubits independants, tous deux actifs. La cause exacte, cote IBM, n'est pas identifiee.
- **Hors de cause**: la machine (3 machines testees), les qubits, le nombre de tirages, les angles, le programme `executor`, les 16 bits classiques.
- **Fausse piste**: la barriere ajoutee par `measure_all()`. La retirer a suffi sur la disposition A, mais pas sur la disposition B apres re-transpilation. Le commit `bb7ce2f` (`RemoveBarriers`) reposait sur cette piste et a ete remplace.

#### Solution appliquee (fonction `executer_sur_ibm`)
- `separer_par_groupe`: decoupe le circuit transpile en un sous-circuit par groupe de qubits relies par des portes. Chaque sous-circuit devient un PUB du meme job. L'atelier de base reste un seul PUB.
- `recoller_tirages`: reconstruit les chaines de 16 bits en associant le i-eme tirage de chaque groupe. C'est valide parce que les groupes sont independants: la distribution complete est le produit des distributions des groupes.
- `delai_max_s` (180 s par defaut): si le job reste en `RUNNING` plus longtemps, il est annule et la cellule leve `TimeoutError` au lieu d'attendre indefiniment.
- Cout: un job annule est quand meme facture ~2 s de QPU; un job x2 termine coute ~4 s.

#### Si un job se bloque a nouveau
- `job.metrics()` donne les horodatages; apres annulation, `circuits_execution_time_ns == 0` confirme que rien n'a ete execute.
- `job._api_client.job_get(job.job_id(), exclude_params=True)` donne l'etat brut (champ `state.reason` s'il existe).
- Comparer `job.inputs["quantum_program"].items` entre un job qui marche et un job bloque, puis signaler au support IBM avec les identifiants de jobs.

</details>

In [35]:
#DEFINITION_RUN
# Preparation de QAOA x2 pour IBM et releve de ses resultats.

# Meme recette que `resoudre_qaoa`, mais retourne le circuit, les angles et la duree (s) pour les envoyer a IBM.
def preparer_qaoa(
    commandes: list[int], paires: list[tuple[int, int]], profondeur: int, nb_departs: int, seed: int = 1
) -> tuple[Any, np.ndarray, float]:
    debut: float = perf_counter()
    hamiltonien, _ = creer_qubo(commandes, paires, max(commandes) + 1).to_ising()
    circuit = QAOAAnsatz(hamiltonien, reps=profondeur).decompose(reps=3)
    cles = tuple(circuit.parameters)
    estimateur = StatevectorEstimator()
    # H n'a que des termes Z et ZZ (diagonale) : simulation numpy directe, ~5x plus rapide que l'estimateur.
    diagonale = hamiltonien.to_matrix(sparse=True).diagonal().real
    n = circuit.num_qubits

    def energie_locale(parametres: np.ndarray) -> float:
        betas, gammas = parametres[:profondeur], parametres[profondeur:]  # ordre de `circuit.parameters`
        psi = np.full(2**n, 2 ** (-n / 2), dtype=complex)
        for beta, gamma in zip(betas, gammas):
            psi = (psi * np.exp(-1j * gamma * diagonale)).reshape((2,) * n)
            c, s = np.cos(beta), np.sin(beta)
            for axe in range(n):
                a0, a1 = np.take(psi, 0, axis=axe), np.take(psi, 1, axis=axe)
                psi = np.stack((c * a0 - 1j * s * a1, -1j * s * a0 + c * a1), axis=axe)
            psi = psi.reshape(-1)
        return float(np.real(np.vdot(psi, diagonale * psi)))

    rng = np.random.default_rng(seed)
    x0 = rng.uniform(0, np.pi, circuit.num_parameters)
    reference = float(estimateur.run([(circuit, hamiltonien, {cles: x0})]).result()[0].data["evs"])
    assert abs(energie_locale(x0) - reference) < 1e-9, "simulation rapide differente de l'estimateur"
    rng = np.random.default_rng(seed)
    essais = [
        minimize(energie_locale, rng.uniform(0, np.pi, circuit.num_parameters), method="COBYLA", options={"maxiter": 300})
        for _ in range(nb_departs)
    ]
    return circuit, min(essais, key=lambda r: r.fun).x, perf_counter() - debut

# Ligne de metriques d'une execution IBM, au format utilise par les graphes.
def releve_ibm(sauvegarde: dict, commandes: list[int], paires: list[tuple[int, int]]) -> dict[str, float]:
    _, profit, part_valide = meilleure_selection_valide(sauvegarde["comptes"], commandes, paires)
    t: dict = sauvegarde["horodatages"]
    tirages: int = sauvegarde["tirages"]
    return {
        "commandes": len(commandes), "tirages": tirages, "valides": round(part_valide * tirages),
        "profit": profit, "duree_ms": 1000 * (_instant(t["finished"]) - _instant(t["running"])).total_seconds(),
        "profondeur": sauvegarde["profondeur"], "portes_2q": sauvegarde["extra"]["portes_2q"],
        "optimum_pct": 100 if profit == recherche_exhaustive(commandes, paires).profit else 0,  # une seule execution
    } | {k: v for k, v in chronologie_ibm(sauvegarde).items() if v is not None}

print("Defini:", preparer_qaoa, releve_ibm)

Defini: <function preparer_qaoa at 0x7e970dfdfb00> <function releve_ibm at 0x7e970dfdfa60>


In [36]:
# QAOA sur IBM Cloud, atelier x2 : on optimise (gamma, beta) sur l'emulateur, puis on echantillonne sur la machine.
RESSOURCES_X2, COMMANDES_X2, PAIRES_X2 = agrandir_atelier(2)
CIRCUIT_X2, PARAMETRES_X2, DUREE_OPTIMISATION_X2 = preparer_qaoa(COMMANDES_X2, PAIRES_X2, PROFONDEUR, nb_departs=3)

print(f"[{DUREE_OPTIMISATION_X2:7.1f} s] optimisation locale terminee")

sauvegarde_x2 = benchmark_qaoa_ibm(
    CIRCUIT_X2, PARAMETRES_X2, COMMANDES_X2, PAIRES_X2, tirages=TIRAGES, machine_ibm=MACHINE_IBM, executer=EXECUTER_SUR_IBM,
    duree_optimisation_s=DUREE_OPTIMISATION_X2, ressources=RESSOURCES_X2,
)
assert sauvegarde_x2 is not None, "passer EXECUTER_SUR_IBM a True pour obtenir les resultats x2"

# Releve au format des graphes : RESULTATS_IBM, plus bas, en garde une copie figee.
print(releve_ibm(sauvegarde_x2, COMMANDES_X2, PAIRES_X2))

[   23.3 s] optimisation locale terminee


[    3.7 s] machine choisie : ibm_fez


[    5.2 s] transpilation terminee : profondeur 250 (circuit ideal : 47)


[    7.1 s] job soumis : dc30d22nj2vs73dhf6ug (2 PUB)


[    7.3 s] statut du job : QUEUED


[    9.6 s] statut du job : RUNNING


[   16.5 s] statut du job : DONE


[   16.8 s] resultat recupere


[   17.1 s] metriques IBM et proprietes de la machine lues


== Variables ==
20 ressources: {'Fraiseuse CNC #1': 'machine', 'Découpeuse laser #1': 'machine', 'Cabine de peinture #1': 'machine', 'Tour #1': 'machine', 'Soudeur certifié #1': 'ouvrier', 'Inspecteur qualité #1': 'ouvrier', 'Électricien #1': 'ouvrier', 'Créneau lundi #1': 'créneau', 'Créneau mercredi #1': 'créneau', 'Créneau vendredi #1': 'créneau', 'Fraiseuse CNC #2': 'machine', 'Découpeuse laser #2': 'machine', 'Cabine de peinture #2': 'machine', 'Tour #2': 'machine', 'Soudeur certifié #2': 'ouvrier', 'Inspecteur qualité #2': 'ouvrier', 'Électricien #2': 'ouvrier', 'Créneau lundi #2': 'créneau', 'Créneau mercredi #2': 'créneau', 'Créneau vendredi #2': 'créneau'}
Profits des 16 commandes: [5, 3, 4, 6, 2, 5, 3, 4, 5, 3, 4, 6, 2, 5, 3, 4]
Paires des 20 conflits: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8), (9, 10), (9, 11), (10, 12), (11, 12), (11, 13), (12, 14), (13, 14), (13, 15), (14, 16), (15, 16)]
== Machine quantique IBM ==
Machine        : ibm

Pour comparaison, une execution precedente de l'atelier x2 sur ibm_fez (memes variables que ci-dessus):

```
== Machine quantique IBM ==
Machine        : ibm_fez (IBM Quantum, cloud)
Processeur     : {'family': 'Heron', 'revision': '2'}
Qubits         : 156
File d'attente : 5 jobs en attente à la soumission
Job            : dc2vh85hdvas73bmua1g
Date           : 2026-11-24T20:50:40.973583Z
Attente reelle : 1.3 s (creation -> debut d'execution)
Statut du job  : DONE (primitive executor, instance crn:v1:bluemix:public:quantum-computing:us-east:a/61e658cacb2547a68a8e83187c424ecd:76c49251-1bf7-4d63-b608-a6a4a057cf10::)
Backend        : version 1.3.37, CLOPS None
Portes de base : cz, id, if_else, measure_2, measure_reset, measure_reset_2, reset_2, rz, sx, x, xslow
Calibration    : 2026-11-24 21:28:26+01:00
T1 / T2 median : 120.1 / 89.9 us
Erreur lecture : 1.18 % (mediane)
Erreur cz : 0.26 % (mediane, porte a 2 qubits)
Portes         : {'rz': 503, 'sx': 480, 'cz': 222, 'x': 36, 'measure': 16, 'barrier': 1} dont 222 a 2 qubits
Qubits utilises: 16 [97, 105, 106, 107, 108, 109, 110, 111, 123, 124, 136, 143, 144, 145, 146, 147]
Fidelite estimee du circuit : 45.69 %
Qiskit 2.5.2, qiskit-ibm-runtime 0.52.1
== Chronologie ==
Optimisation (emulateur, local) : 26.3 s
Transpilation (local)           : 1.3 s
File d'attente IBM              : 1.3 s
Execution IBM                   : 8.9 s
Soumission -> resultat (local)  : 14.7 s
Total de bout en bout           : 42.3 s
== Resultats ==
commandes [1, 4, 5, 8, 9, 12, 13, 16], profit 34
29.6 % des tirages valides
4096 tirages par exécution
Temps d'exécution sur IBM : 8921.652 ms
```

In [37]:
#DEFINITION_RUN
# Graphes : QAOA sur machine IBM, atelier de base et atelier x2. Reutilise graphe_metrique et graphe_tirages.
# Valeurs relevees dans les resultats IBM ci-dessus (ibm_fez, meme session et meme calibration, 1 execution chacun, cellules d'execution IBM).
# La duree ne compte que l'echantillonnage sur IBM : les angles viennent de l'emulateur.
RESULTATS_IBM: dict[str, dict[str, float]] = {
    "Base (8 commandes)": {
        "commandes": 8, "tirages": 4096, "valides": round(0.506 * 4096),
        "profit": 17, "duree_ms": 4512.083, "profondeur": 278, "portes_2q": 108,
        "fidelite_pct": 69.81, "optimum_pct": 100,
    },
    "Atelier x2 (16 commandes)": {
        "commandes": 16, "tirages": 4096, "valides": round(0.266 * 4096),
        "profit": 34, "duree_ms": 5982.218, "profondeur": 250, "portes_2q": 222,
        "fidelite_pct": 46.06, "optimum_pct": 100,
    },
}

# Facteur de croissance de chaque metrique entre l'atelier de base et l'atelier x2.
def graphe_croissance_ibm(ax, base: dict[str, float], agrandi: dict[str, float]) -> None:
    metriques: dict[str, str] = {
        "commandes": "Commandes",
        "profit": "Profit",
        "portes_2q": "Portes à 2 qubits",
        "fidelite_pct": "Fidélité estimée",
        "duree_ms": "Durée sur IBM",
    }
    facteurs: list[float] = [agrandi[c] / base[c] for c in metriques]
    barres = ax.barh(list(metriques.values()), facteurs, color="#E45756")
    ax.bar_label(barres, labels=[f"x{f:.4g}" for f in facteurs], padding=3)
    ax.set_xscale("log")
    ax.set_title("Croissance de base à x2")
    ax.margins(x=0.2)

def graphe_ibm(resultats: dict[str, dict[str, float]]) -> None:
    base, agrandi = list(resultats.values())
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    graphe_metrique(axes[0, 0], resultats, "duree_ms", "Durée d'exécution sur IBM (échelle log)", "ms", log=True)
    graphe_tirages(axes[0, 1], resultats)
    graphe_metrique(axes[1, 0], resultats, "fidelite_pct", "Fidélité estimée du circuit transpilé", "%")
    graphe_croissance_ibm(axes[1, 1], base, agrandi)
    fig.suptitle("QAOA sur machine IBM : atelier de base contre atelier x2", fontsize=14)
    fig.tight_layout()
    plt.show()

print("Defini:", graphe_croissance_ibm, graphe_ibm)

Defini: <function graphe_croissance_ibm at 0x7e970dfded40> <function graphe_ibm at 0x7e970dfdfec0>


In [38]:
graphe_ibm(RESULTATS_IBM)

/tmp/ipykernel_1278368/562974804.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
#DEFINITION_RUN
# Graphes : QAOA sur emulateur contre QAOA sur machine IBM, atelier de base et atelier x2.
# Les cellules d'execution demandent COMPARAISON (emulateur et exhaustive) et RESULTATS_IBM.
COULEURS_COMP: dict[str, str] = {"Exhaustive": "#4C78A8", "QAOA émulé": "#F58518", "QAOA IBM": "#B279A2"}

# Ligne au format COMPARAISON : une seule execution sur IBM, donc sans ecart-type.
def ligne_comparaison(r: dict[str, float]) -> dict[str, float]:
    return {
        "duree_ms": r["duree_ms"], "ecart_ms": 0.0,
        "valide_pct": 100 * r["valides"] / r["tirages"], "optimum_pct": r["optimum_pct"],
    }

# Lu a l'appel : definir cette cellule ne demande ni COMPARAISON ni RESULTATS_IBM.
def comparaison_ibm() -> dict[str, dict[str, float]]:
    return dict(zip(COMPARAISON["QAOA"], map(ligne_comparaison, RESULTATS_IBM.values())))

# Barres groupees : un groupe par atelier, une barre par algorithme.
def barres_groupees_de(ax, comparaison: dict[str, dict[str, dict[str, float]]], cle: str, titre: str, unite: str,
                       log: bool = False, erreur: str | None = None) -> None:
    ateliers: list[str] = list(next(iter(comparaison.values())))
    largeur: float = 0.38
    for k, (algo, donnees) in enumerate(comparaison.items()):
        x = [i + (k - 0.5) * largeur for i in range(len(ateliers))]
        valeurs: list[float] = [donnees[a][cle] for a in ateliers]
        yerr = [donnees[a][erreur] for a in ateliers] if erreur else None
        barres = ax.bar(x, valeurs, largeur, label=algo, color=COULEURS_COMP[algo], yerr=yerr, capsize=4)
        ax.bar_label(barres, labels=[(f"{v:,.0f}" if v >= 1000 else f"{v:.4g}") + f" {unite}".rstrip() for v in valeurs], padding=3, fontsize=8)
    ax.set_xticks(range(len(ateliers)), ateliers)
    ax.set_title(titre)
    ax.legend()
    if log:
        ax.set_yscale("log")
    ax.margins(y=0.2)

# Facteur de croissance base -> x2 de la duree et de la part valide, par algorithme.
def croissance_de(ax, comparaison: dict[str, dict[str, dict[str, float]]]) -> None:
    metriques: dict[str, str] = {"duree_ms": "Durée", "valide_pct": "Part valide"}
    largeur: float = 0.38
    for k, (algo, donnees) in enumerate(comparaison.items()):
        base, agrandi = list(donnees.values())
        facteurs: list[float] = [agrandi[m] / base[m] for m in metriques]
        barres = ax.bar([i + (k - 0.5) * largeur for i in range(len(metriques))], facteurs, largeur, label=algo, color=COULEURS_COMP[algo])
        ax.bar_label(barres, labels=[f"x{f:.3g}" for f in facteurs], padding=3)
    ax.set_xticks(range(len(metriques)), list(metriques.values()))
    ax.set_yscale("log")
    ax.set_title("Croissance de base à x2 (échelle log)")
    ax.legend()
    ax.margins(y=0.2)

def graphe_comparaison_de(comparaison: dict[str, dict[str, dict[str, float]]], titre: str) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    barres_groupees_de(axes[0, 0], comparaison, "duree_ms", "Durée par exécution (échelle log)", "ms", log=True, erreur="ecart_ms")
    croissance_de(axes[0, 1], comparaison)
    barres_groupees_de(axes[1, 0], comparaison, "valide_pct", "Part de choix valides / tirages valides", "%")
    barres_groupees_de(axes[1, 1], comparaison, "optimum_pct", "Optimum trouvé", "%")
    axes[1, 1].set_ylim(0, 130)
    fig.suptitle(titre, fontsize=14)
    fig.tight_layout()
    plt.show()

print("Defini:", ligne_comparaison, comparaison_ibm, barres_groupees_de, croissance_de, graphe_comparaison_de)

Defini: <function ligne_comparaison at 0x7e9704f13060> <function comparaison_ibm at 0x7e9704f12d40> <function barres_groupees_de at 0x7e9704f12840> <function croissance_de at 0x7e9704f12480> <function graphe_comparaison_de at 0x7e9704f12f20>


In [40]:
# Duree emulee = optimisation COBYLA + echantillonnage ; duree IBM = echantillonnage seul.
graphe_comparaison_de(
    {"QAOA émulé": COMPARAISON["QAOA"], "QAOA IBM": comparaison_ibm()},
    "QAOA sur émulateur contre QAOA sur machine IBM",
)

/tmp/ipykernel_1278368/3262853903.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# Graphes : recherche exhaustive locale contre QAOA sur machine IBM. Reutilise graphe_comparaison_de.
graphe_comparaison_de(
    {"Exhaustive": COMPARAISON["Exhaustive"], "QAOA IBM": comparaison_ibm()},
    "Recherche exhaustive locale contre QAOA sur machine IBM",
)

/tmp/ipykernel_1278368/3262853903.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusion

Nous avons resolu le meme probleme d'atelier (MWIS ecrit en QUBO, 8 puis 16 commandes) de trois facons, dans une seule execution du notebook, sur la meme machine locale (Intel Core Ultra 7 155H, 14.6 Go) et, pour QAOA natif, sur le processeur IBM Heron r2 ibm_fez (156 qubits, meme calibration pour les deux ateliers):

1. **Recherche exhaustive**: teste les $2^n$ selections, en Python, sur un seul coeur.
2. **QAOA emule**: circuit $p = 3$, 3 departs COBYLA, 4096 tirages, simule exactement (vecteur d'etat, sans bruit) sur le processeur classique.
3. **QAOA natif**: memes angles $(\gamma, \beta)$ que l'emulateur, seul l'echantillonnage (4096 tirages) est execute sur la machine IBM.

### Tableau recapitulatif

| Indicateur | Exhaustive x1 | Exhaustive x2 | QAOA emule x1 | QAOA emule x2 | QAOA IBM x1 | QAOA IBM x2 |
|---|---|---|---|---|---|---|
| Commandes / qubits | 8 / n/a | 16 / n/a | 8 / 8 | 16 / 16 | 8 / 8 | 16 / 16 |
| Profit trouve | 17 | 34 | 17 | 34 | 17 | 34 |
| Optimum trouve | garanti | garanti | 8/10 | 3/3 | 1/1 | 1/1 |
| Choix valides (espace) / tirages valides | 16.0 % | 2.6 % | 62.9 % | 47.0 % | 50.6 % | 26.6 % |
| Duree par execution | 0.123 ms | 27.9 ms | 4.88 s (± 0.36 s) | 112.6 s (± 3.3 s) | 4.51 s * | 5.98 s * |
| Duree de bout en bout | 0.123 ms | 27.9 ms | 4.88 s | 112.6 s | 16.2 s ** | 36.5 s ** |
| Temps CPU local | 0.118 ms | 26.7 ms | 3.1 s | 857.0 s (761 % d'un coeur) | n/a (QPU) | n/a (QPU) |
| Profondeur du circuit | n/a | n/a | 47 | 47 | 278 | 250 |
| Portes a 2 qubits | n/a | n/a | 60 | 120 | 108 | 222 |
| Fidelite estimee | n/a | n/a | 100 % (sans bruit) | 100 % (sans bruit) | 69.8 % | 46.1 % |

\* Echantillonnage seul sur IBM: l'optimisation des angles est faite sur l'emulateur.

\*\* Optimisation locale + transpilation + soumission jusqu'au resultat (file d'attente et execution comprises): 5.0 + 1.2 + 10.1 s pour l'atelier de base (job `dc30cots2omc738cnhn0`), 23.3 + 1.6 + 11.6 s pour l'atelier x2 (job `dc30d22nj2vs73dhf6ug`). L'atelier de base est optimise avec l'estimateur Qiskit, l'atelier x2 avec la simulation numpy de `preparer_qaoa`.

Les profondeurs incluent la mesure (46 sans elle). Le circuit ideal compte 2 portes CX par paire en conflit et par couche: $10 \times 3 \times 2 = 60$, puis $20 \times 3 \times 2 = 120$. Le circuit x2 n'est pas plus profond car ses deux ateliers independants s'executent en parallele.

### 1. Qualite des solutions

- **Les trois methodes trouvent le meme optimum**: commandes [1, 4, 5, 8] (profit 17) sur l'atelier de base, [1, 4, 5, 8, 9, 12, 13, 16] (profit 34) sur l'atelier x2.
- **Seule la recherche exhaustive le garantit.** QAOA emule l'a manque 2 fois sur 10 sur l'atelier de base (probablement un minimum local de COBYLA), soit 11 succes sur 13 executions au total (85 %). QAOA natif l'a trouve a chaque execution affichee dans ce notebook (2 sur l'atelier de base, ibm_fez et ibm_marrakesh; 2 sur l'atelier x2, ibm_fez), mais avec les angles deja optimises par l'emulateur.
- **Sur 8 commandes, trouver l'optimum ne prouve rien**: avec 4096 tirages pour seulement $2^8 = 256$ selections, un tirage uniforme au hasard tomberait sur l'optimum avec une probabilite de $1 - (255/256)^{4096} \approx 99.99999\,\%$.
- **Sur 16 commandes, le resultat devient significatif**: il y a $2^{16} = 65\,536$ selections, et un tirage uniforme de 4096 echantillons n'a que $1 - (65\,535/65\,536)^{4096} \approx 6.1\,\%$ de chances de contenir l'optimum. QAOA l'a trouve 5 fois sur 5 (3 emule, 2 natif): le circuit oriente bien la mesure vers la bonne reponse.

### 2. Concentration sur les selections valides

La part de tirages sans conflit mesure a quel point le circuit "comprend" les contraintes, par rapport au hasard (part de selections valides dans l'espace de recherche):

| | Hasard | QAOA emule | QAOA IBM |
|---|---|---|---|
| Atelier x1 | 16.0 % | 62.9 % (x3.9) | 50.6 % (x3.2) |
| Atelier x2 | 2.6 % | 47.0 % (x18.3) | 26.6 % (x10.4) |

- QAOA concentre la mesure sur les selections valides, et cet effet **grandit** avec la taille du probleme (x3.9 puis x18.3 en emule, x3.2 puis x10.4 sur IBM).
- Le bruit materiel en efface une partie: la machine IBM conserve 80 % de la part valide de l'emulateur sur 8 qubits (50.6 / 62.9), mais seulement 57 % sur 16 qubits (26.6 / 47.0).
- Cette degradation suit la fidelite estimee du circuit transpile: 69.8 % puis 46.1 % (x0.66), quand le nombre de portes a 2 qubits double (108 puis 222, x2.06).
- Les executions IBM varient d'une machine et d'une calibration a l'autre: 50.6 % (ibm_fez) et 47.8 % (ibm_marrakesh) sur l'atelier de base, 26.6 % et 29.6 % (ibm_fez, deux calibrations) sur l'atelier x2.

### 3. Temps d'execution

| Rapport de durees | Atelier x1 | Atelier x2 |
|---|---|---|
| QAOA emule / exhaustive | x39 700 | x4 040 |
| QAOA IBM (echantillonnage) / exhaustive | x36 700 | x215 |
| QAOA IBM (de bout en bout) / exhaustive | x132 000 | x1 310 |
| QAOA emule / QAOA IBM (echantillonnage) | x1.08 | x18.8 |
| QAOA emule / QAOA IBM (de bout en bout) | x0.30 | x3.1 |

- **A cette taille, la recherche exhaustive gagne sans discussion**: 27.9 ms pour 65 536 candidats, soit environ 0.43 µs par candidat.
- **Sur l'echantillonnage seul, QAOA natif rattrape puis depasse QAOA emule**: presque a egalite a 8 qubits (x1.08), 18.8 fois plus rapide a 16 qubits.
- **De bout en bout, le gagnant change avec la taille**: sur 8 commandes, l'emulateur est 3.3 fois plus rapide (4.9 s contre 16.2 s), car l'aller-retour vers IBM coute plus que la simulation de 8 qubits. Sur 16 commandes, IBM devient 3.1 fois plus rapide (36.5 s contre 112.6 s).
- Une part importante du temps IBM n'est pas du calcul: transpilation, envoi et recuperation ajoutent environ 5 s, la file d'attente a dure 1.1 a 2.1 s, et l'execution de 4096 tirages prend 4.5 a 8.9 s pour un circuit d'environ 7 µs.

### 4. Passage a l'echelle (de 8 a 16 commandes)

| Croissance base -> x2 | Exhaustive | QAOA emule | QAOA IBM |
|---|---|---|---|
| Duree | x227 | x23 | x1.33 (echantillonnage), x2.25 (de bout en bout) |
| Temps CPU local | x226 | x276 | n/a |
| Ressource qui grandit | candidats: x256 ($2^8$) | evaluations: x1.03 (871 -> 894) | portes 2 qubits: x2.06 |

- **Recherche exhaustive**: la duree suit $2^n$ (x227 pour x256 candidats). A ~0.43 µs par candidat, on peut extrapoler: ~30 minutes pour 32 commandes, ~5 jours pour 40, ~15 ans pour 50.
- **QAOA emule**: le nombre d'evaluations de l'energie reste quasi constant (x1.03), mais chaque evaluation simule un vecteur de $2^n$ amplitudes. Le temps CPU suit donc une croissance exponentielle (x276), masquee en temps reel par le parallelisme (761 % d'un coeur). En memoire, un vecteur d'etat coute $16 \times 2^n$ octets: 8 Go a 29 qubits, 16 Go a 30 qubits. **Sur notre machine de 14.6 Go, l'emulation s'arrete vers 29 commandes**, la ou la recherche exhaustive ne prendrait encore que ~4 minutes.
- **QAOA natif**: l'echantillonnage n'augmente que de x1.33 et le circuit reste aussi profond (278 puis 250). Le cout reel se deplace vers le **bruit**: rapportee aux portes a 2 qubits, la fidelite estimee perd ~0.33 % par porte. Elle predit 47.8 % pour l'atelier x2, proche des 46.1 % mesures. En extrapolant, elle tomberait a ~23 % pour 32 commandes (444 portes) et ~5 % pour 64 commandes (888 portes): la part de tirages valides serait alors proche du hasard.

### 5. Couts pratiques

- **Recherche exhaustive**: une vingtaine de lignes de Python, aucune dependance, resultat exact et reproductible.
- **QAOA emule**: necessite la chaine QUBO -> Hamiltonien -> circuit -> COBYLA, plusieurs departs pour eviter les minima locaux, et un reglage de $p$, de la penalite $A$ et du nombre de tirages. Le resultat varie avec la graine.
- **QAOA natif**: ajoute un compte IBM (temps machine mensuel limite sur le plan Open), la transpilation (profondeur x5.9: 47 -> 278), l'attente en file et des comportements imprevus: l'atelier x2 envoye d'un seul bloc restait bloque en `RUNNING` sur 3 machines, et il a fallu le decouper en 2 PUB. Les angles viennent toujours de l'emulateur: l'optimisation sur le materiel (~900 evaluations par execution) n'a pas ete mesuree.

### Bilan

| | Recherche exhaustive | QAOA emule | QAOA natif (IBM) |
|---|---|---|---|
| Optimum | Garanti | Probable (85 % des executions) | Trouve 4/4, angles fournis par l'emulateur |
| Vitesse a 8-16 commandes | **Meilleure** (0.12 ms, 28 ms) | 4.9 s, 113 s | 16.2 s, 36.5 s de bout en bout (4.5 s, 6.0 s d'echantillonnage) |
| Croissance mesuree | $2^n$ (x227) | $2^n$ en CPU (x276) | Quasi lineaire (x1.33 en echantillonnage) |
| Limite | Temps: ~15 ans a 50 commandes | Memoire: ~29 commandes sur 14.6 Go | Bruit: fidelite ~5 % vers 64 commandes |

- **Pour l'atelier etudie, la recherche exhaustive est le bon choix**: elle est exacte, 215 a 132 000 fois plus rapide que QAOA, et simple.
- **QAOA emule n'apporte aucun gain de performance**: simuler un ordinateur quantique coute au moins autant que la force brute ($2^n$). Il sert a concevoir et valider le circuit, et a trouver les angles. Sur 8 commandes, il reste meme plus rapide que l'aller-retour vers IBM.
- **QAOA natif est la seule variante dont le cout n'explose pas avec $n$**: des 16 commandes, il bat l'emulateur de bout en bout (x3.1), et il conserve la bonne reponse malgre une fidelite de 46.1 %. Mais il reste plus lent que la force brute a cette taille, et le bruit croit avec le nombre de portes.
- **Aucun avantage quantique n'est demontre ici.** Il faudrait un atelier assez grand pour que la force brute ne soit plus praticable (plusieurs dizaines de commandes), des machines moins bruitees ou de la correction d'erreurs, et une optimisation des angles realisee sur le materiel lui-meme. Il faudrait aussi comparer QAOA a des solveurs classiques specialises (programmation lineaire en nombres entiers, heuristiques MWIS), bien plus rapides que la force brute.

## Glossaire

### Exposition du probleme et traduction mathematique

- **Ressource**: element de l'atelier (machine, ouvrier, creneau) utilisable par une seule commande a la fois.
- **Conflit**: deux commandes qui ont besoin d'une meme ressource et ne peuvent pas etre acceptees ensemble.
- **Graphe de conflits**: graphe dont les sommets sont les commandes et les aretes les conflits.
- **MWIS (Maximum Weight Independent Set)**: choisir des sommets sans voisins entre eux, avec un poids total maximal. Ici, le poids est le profit.
- **NP-Hard**: classe de problemes pour lesquels aucun algorithme rapide (polynomial) n'est connu.

### Definition des variables et recherche exhaustive

- **Recherche exhaustive (force brute)**: teste les $2^n$ selections possibles et garde la meilleure valide.
- **Selection valide**: ensemble de commandes sans aucun conflit.
- **Benchmark**: mesure repetee des performances d'un algorithme sur la meme machine.
- **Temps CPU (utilisateur / systeme)**: temps de calcul du processeur, dans le programme ou dans le noyau.
- **Pic de memoire**: memoire maximale utilisee pendant l'execution.
- **Changement de contexte**: moment ou le processeur passe a un autre processus (volontaire ou force).

### QAOA et modele mathematique

- **QAOA (Quantum Approximate Optimization Algorithm)**: algorithme hybride qui cherche une solution approchee d'un probleme d'optimisation combinatoire.
- **Algorithme hybride quantique-classique**: une partie quantique prepare et mesure un etat, une partie classique ajuste ses parametres.
- **Qubit**: unite d'information quantique, superposition de $|0\rangle$ et $|1\rangle$. Ici, un qubit par commande.
- **Superposition**: etat d'un qubit combinant $|0\rangle$ et $|1\rangle$, avec des amplitudes.
- **Amplitude**: nombre complexe dont le carre du module donne la probabilite d'une mesure.
- **Penalite ($A$)**: coefficient qui rend un conflit plus couteux que le profit qu'il rapporte.
- **QUBO (Quadratic Unconstrained Binary Optimization)**: minimisation d'une fonction quadratique de variables binaires, sans contrainte.
- **Hamiltonien**: operateur dont les valeurs propres sont les energies. Ici, il encode le cout $f(x)$.
- **Modele d'Ising**: Hamiltonien fait de termes $Z_i$ et $Z_i Z_j$.
- **Operateurs de Pauli ($X$, $Z$)**: portes a un qubit. $Z$ lit le bit (valeur propre $+1$ ou $-1$), $X$ l'inverse.
- **Etat fondamental**: etat d'energie minimale. Ici, la meilleure selection.
- **Circuit parametre**: circuit quantique dont les portes dependent de parametres reels ($\gamma$, $\beta$).
- **Profondeur ($p$)**: nombre de couches $U_C$, $U_B$ du circuit.
- **Hadamard**: porte qui met un qubit dans l'etat $|+\rangle$.
- **Valeur moyenne $\langle H \rangle$**: energie moyenne de l'etat prepare, estimee par mesures.
- **Tirage (shot)**: une mesure du circuit. On en fait plusieurs pour estimer des probabilites.
- **COBYLA**: optimiseur classique qui n'utilise pas de gradient.
- **Minimum local**: point ou l'energie est plus basse qu'autour, mais pas la plus basse possible.

### Implementation

- **Qiskit / Qiskit optimization**: bibliotheques Python pour les circuits quantiques et les problemes d'optimisation.
- **`QuadraticProgram`**: objet Qiskit qui decrit un probleme quadratique (variables, termes lineaires et quadratiques).
- **`QAOAAnsatz`**: objet Qiskit qui construit le circuit QAOA a partir d'un Hamiltonien.
- **Estimator**: primitive Qiskit qui calcule $\langle H \rangle$.
- **Sampler**: primitive Qiskit qui mesure le circuit et renvoie les comptes.
- **Simulation par vecteur d'etat**: calcul exact de l'etat quantique sur un processeur classique. Son cout double a chaque qubit.
- **Graine (seed)**: valeur qui fixe le tirage aleatoire pour reproduire un resultat.

### Execution sur une machine IBM

- **QPU (Quantum Processing Unit)**: processeur quantique physique, ici un IBM Heron r2.
- **Transpilation**: reecriture du circuit avec les portes natives et la topologie de la machine. Elle augmente sa profondeur.
- **Porte CZ**: porte native a 2 qubits des machines IBM Heron, principale source d'erreur du circuit.
- **PUB (Primitive Unified Bloc)**: un circuit et ses parametres, envoyes ensemble a une primitive dans un job.
- **File d'attente**: temps entre la soumission du job et le debut de son execution.
- **Calibration**: mesure periodique des erreurs de la machine. Elle peut changer les qubits physiques choisis par la transpilation.
- **T1 / T2**: temps de relaxation et de decoherence d'un qubit. Au-dela, l'etat quantique est perdu.
- **Erreur de lecture**: probabilite de mesurer le mauvais bit.
- **Fidelite estimee**: produit des $(1 - \text{erreur})$ de chaque porte et mesure, soit la probabilite approchee qu'aucune erreur ne se produise.
- **Bruit**: ensemble des erreurs materielles qui ecartent les tirages de la simulation ideale.

### Interpretation des resultats

- **Deterministe / stochastique**: un algorithme deterministe donne toujours le meme resultat, un algorithme stochastique non.
- **Optimum**: meilleure solution valide, de profit maximal.
- **Avantage quantique**: cas ou un algorithme quantique resout un probleme plus vite que le meilleur algorithme classique.
